# Estructura de carga y limpieza de datos

Para garantizar un flujo de trabajo coherente, organizado y reproducible, cada conjunto de datos de NHANES se procesa de forma independiente antes de fusionarse con el conjunto de datos final.

Cada sección de preparación de datos sigue la misma estructura, lo que permite que cada conjunto de datos quede completamente limpio y listo para su análisis por sí solo. Este enfoque mejora la legibilidad del código, simplifica la depuración y hace que todo el proceso de depuracion sea más fácil de entender y mantener.

Cada conjunto de datos se procesa siguiendo los siguientes pasos:

1. **Cargar el conjunto de datos** :
   Importar el conjunto de datos original de NHANES.

2. **Renombrar variables** :
   Sustituir los códigos de variables de NHANES por nombres de columna descriptivos y significativos.

3. **Seleccionar variables** :
   Conservar únicamente las variables relevantes para los objetivos de este proyecto.

4. **Sustituir los códigos de valores perdidos de NHANES por NaN** :
   Convertir los códigos de valores perdidos específicos de NHANES (p. ej., *Refused*, *Don't know*, etc.) en valores perdidos estándar (`NaN`) según el libro de códigos oficial.

5. **Correcciones específicas del conjunto de datos** :
   Se aplica las correcciones necesarias para ese conjunto de datos concreto, como corregir valores importados incorrectamente o gestionar casos especiales identificados durante la exploración de los datos.

6. **Decodifica las variables (cuando proceda)** :
   Convierte los valores codificados (por ejemplo, cadenas de bytes que representan la hora) a formatos legibles.

7. **Sustituir los valores codificados por etiquetas descriptivas** :
   Sustituir los códigos numéricos de categoría por etiquetas legibles para mejorar la interpretabilidad y facilitar el análisis exploratorio de los datos.

Seguir este flujo de trabajo de preprocesamiento estandarizado garantiza que cada conjunto de datos esté limpio, sea coherente y esté listo para integrarse en el conjunto de datos analítico final.


In [ ]:
#1 CARGA Y LIMPIEZA ARCHIVO DEMO_J 
import pandas as pd
import numpy as np

demo=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\DEMO_J.xpt")

# ===========================
# Rename variables
# ===========================

demo=demo.rename(columns={'RIAGENDR':'Gender',
                                'RIDAGEYR':'Age',
                                'RIDRETH3':'Race',
                                'DMDEDUC2':'Education',
                                'DMDMARTL':'Marital_Status',
                                'INDFMPIR':'Poverty_Index'}) #Annual family income

# Select variables
df_demo=demo[['SEQN','Gender','Age','Race','Education','Marital_Status','Poverty_Index']].copy()


# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

categorical_columns = [
    "Education",
    "Marital_Status"]

df_demo[categorical_columns] = df_demo[categorical_columns].replace({
    7: np.nan,
    9: np.nan,
    77: np.nan,
    99: np.nan
})

# ==========================================
# Correct anomalous value imported by read_sas()
# ==========================================

# read_sas() occasionally imports SAS value 0 as 5.397605346934028e-79

df_demo["Age"] = (
    df_demo["Age"]
    .replace(5.397605346934028e-79, 0)
)

df_demo["Poverty_Index"] = (
    df_demo["Poverty_Index"]
    .replace(5.397605346934028e-79, 0)
)

# ===========================
# Replace coded values with descriptive labels
# ===========================

#Gender:
df_demo['Gender']=df_demo['Gender'].replace({1:'Male',2:'Female'})

#RACE:
df_demo["Race"] = df_demo["Race"].replace({
    1: "Mexican American",
    2: "Other Hispanic",
    3: "Non-Hispanic White",
    4: "Non-Hispanic Black",
    6: "Non-Hispanic Asian",
    7: "Other / Multi-Racial"
})

# Education
df_demo["Education"] = df_demo["Education"].replace({
    1: "Less than 9th grade",
    2: "9-11th grade",
    3: "High school/GED",
    4: "Some college/AA degree",
    5: "College graduate or above"
})

# Marital status
df_demo["Marital_Status"] = df_demo["Marital_Status"].replace({
    1: "Married",
    2: "Widowed",
    3: "Divorced",
    4: "Separated",
    5: "Never married",
    6: "Living with partner"
})


df_demo.head()


In [ ]:
#2 CARGA Y LIMPIEZA ARCHIVO SLQ_J (sleep disorders)
sleep=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\SLQ_J.xpt")

sleep = sleep.rename(columns={
    'SLQ300': 'Bedtime_Workdays',
    'SLQ310': 'Wakeup_Workdays',
    'SLD012': 'Sleep_Hours_Workdays',

    'SLQ320': 'Bedtime_Weekend',
    'SLQ330': 'Wakeup_Weekend',
    'SLD013': 'Sleep_Hours_Weekend',

    'SLQ030': 'Snoring_Frequency',
    'SLQ040': 'Breathing_Interruptions_Frequency',
    'SLQ050': 'Reported_Sleep_Trouble_To_Doctor',
    'SLQ120': 'Daytime_Sleepiness_Frequency'
})


df_sleep = sleep[[
    'SEQN',

    'Bedtime_Workdays',
    'Wakeup_Workdays',
    'Sleep_Hours_Workdays',

    'Bedtime_Weekend',
    'Wakeup_Weekend',
    'Sleep_Hours_Weekend',

    'Snoring_Frequency',
    'Breathing_Interruptions_Frequency',
    'Reported_Sleep_Trouble_To_Doctor',
    'Daytime_Sleepiness_Frequency'
]].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

categorical_columns = [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Reported_Sleep_Trouble_To_Doctor",
    "Daytime_Sleepiness_Frequency"
]

df_sleep[categorical_columns] = df_sleep[categorical_columns].replace({
    7: np.nan,
    9: np.nan
})

# ===========================
# Decode time variables
# ===========================

time_columns = [
    "Bedtime_Workdays",
    "Wakeup_Workdays",
    "Bedtime_Weekend",
    "Wakeup_Weekend"]

for col in time_columns: #Transformamos el tipo de objeto bytes de las columnas de tiempo #We convert the byte object type of the time columns
    df_sleep[col] = df_sleep[col].str.decode("utf-8")
    
# ===========================
# Correct incorrectly imported SAS value
# ===========================

# read_sas() occasionally imports some SAS zero values as 5.397605346934028e-79
for col in [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Daytime_Sleepiness_Frequency"
]:
    df_sleep[col] = df_sleep[col].replace(5.397605346934028e-79, 0)    
    

    
# ===========================
# Replace coded values with descriptive labels
# ===========================

frequency_labels_sleep = {
    0: "Never",
    1: "Rarely (1-2 nights/week)",
    2: "Occasionally (3-4 nights/week)",
    3: "Frequently (5+ nights/week)"
}

for col in [
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency"
]:
    df_sleep[col] = df_sleep[col].replace(frequency_labels_sleep)

daytime_sleepiness_labels = {
    0: "Never",
    1: "Rarely (1 time/month)",
    2: "Sometimes (2-4 times/month)",
    3: "Often (5-15 times/month)",
    4: "Almost always (16-30 times/month)"
}

df_sleep['Daytime_Sleepiness_Frequency'] = df_sleep['Daytime_Sleepiness_Frequency'].replace(daytime_sleepiness_labels)

df_sleep["Reported_Sleep_Trouble_To_Doctor"] = (
    df_sleep["Reported_Sleep_Trouble_To_Doctor"]
    .replace({
        1: "Yes",
        2: "No"
    })
)


df_sleep.head()



In [ ]:
#3 LIMPIEZA ARCHIVO BMX_J (BODY MEASURES)

# ===========================
# Load dataset
# ===========================

body=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\BMX_J.xpt")

# ===========================
# Rename variables
# ===========================

body = body.rename(columns={
    "BMXWT": "Weight",
    "BMXHT": "Height",
    "BMXBMI": "BMI",
    "BMXWAIST": "Waist_Circumference",
    "BMXHIP": "Hip_Circumference"
})

# ===========================
# Select variables
# ===========================

df_body = body[[
    'SEQN',
    "Weight",
    "Height",
    "BMI",
    "Waist_Circumference",
    "Hip_Circumference"
]].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

# No replacement needed.
# Missing values are already imported as NaN.


df_body.head()


In [ ]:
#5 LIMPIEZA ARCHIVO Glycohemoglobin (GHB_J)

# ===========================
# Load dataset
# ===========================

df_ghb=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\GHB_J.xpt")


# ===========================
# Rename variables
# ===========================
df_ghb = df_ghb.rename(columns={
    'LBXGH': 'HbA1c' #Es una medida que indica el nivel medio de glucosa en sangre durante los últimos 2-3 meses. 
                     #This is a measure that indicates the average blood glucose level over the past 2–3 months.
})

# ===========================
# Select variables
# ===========================
df_ghb = df_ghb[['SEQN', 'HbA1c']].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

# No replacement needed.
# Missing values are already imported as NaN.

df_ghb
    

In [ ]:
#6 LIMPIEZA ARCHIVO Blood Pressure (BPX_J)

# ===========================
# Load dataset
# ===========================
pressure=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\BPX_J.xpt")

#reemplazando el valor erroneo del archivo 
# read_sas() occasionally imports some SAS zero values as 5.397605346934028e-79
for col in ["BPXDI1", "BPXDI2", "BPXDI3", "BPXDI4"]:
    pressure[col] = pressure[col].replace(5.397605346934028e-79, 0)

# ===========================
# Rename variables
# ===========================
pressure = pressure.rename(columns={
    'BPXSY1': 'Systolic_BP_1',
    'BPXSY2': 'Systolic_BP_2',
    'BPXSY3': 'Systolic_BP_3',
    'BPXSY4': 'Systolic_BP_4',

    'BPXDI1': 'Diastolic_BP_1',
    'BPXDI2': 'Diastolic_BP_2',
    'BPXDI3': 'Diastolic_BP_3',
    'BPXDI4': 'Diastolic_BP_4'
})

# ===========================
# Select variables
# ===========================
df_pressure = pressure[[
    'SEQN',
    'Systolic_BP_1',
    'Systolic_BP_2',
    'Systolic_BP_3',
    'Systolic_BP_4',
    'Diastolic_BP_1',
    'Diastolic_BP_2',
    'Diastolic_BP_3',
    'Diastolic_BP_4'
]].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

# No replacement needed.
# Missing values are already imported as NaN.


# ===========================
# Calculate mean blood pressure
# ===========================

# Calculate the mean systolic blood pressure from the four available measurements.
df_pressure["Systolic_BP"] = df_pressure[
    ["Systolic_BP_1","Systolic_BP_2","Systolic_BP_3","Systolic_BP_4"]
].mean(axis=1)

# Calculate the mean diastolic blood pressure from the four available measurements.
df_pressure["Diastolic_BP"] = df_pressure[
    ["Diastolic_BP_1","Diastolic_BP_2","Diastolic_BP_3","Diastolic_BP_4"]
].mean(axis=1)


# ===========================
# Keep final variables
# ===========================

# Keep only the participant identifier and the mean systolic and diastolic
# blood pressure values for the final analytical dataset.
df_pressure=df_pressure[['SEQN','Systolic_BP','Diastolic_BP']].copy()


df_pressure.head()

In [ ]:
#7 LIMPIEZA ARCHIVO Cholesterol - High - Density Lipoprotein (HDL) (HDL_J)

# ===========================
# Load dataset
# ===========================
df_hdl=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\HDL_J.xpt")


# ===========================
# Rename variables
# ===========================
df_hdl = df_hdl.rename(columns={
    'LBDHDD': 'HDL' #HDL significa High-Density Lipoprotein, conocido como colesterol HDL o "colesterol bueno".
})

# ===========================
# Select variables
# ===========================
df_hdl = df_hdl[['SEQN', 'HDL']].copy()

df_hdl.head()

In [ ]:
#8 LIMPIEZA ARCHIVO Physical Activity (PAQ_J)
import pandas as pd

# ===========================
# Load dataset
# ===========================

df_paq=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\PAQ_J.xpt")

df_paq = df_paq.rename(columns={
    # Actividad física en el trabajo 
    # Physical Activity at Work
    'PAQ605': 'Vigorous_Work',
    'PAQ610': 'Vigorous_Work_Days',
    'PAD615': 'Vigorous_Work_Minutes',

    'PAQ620': 'Moderate_Work',
    'PAQ625': 'Moderate_Work_Days',
    'PAD630': 'Moderate_Work_Minutes',

    # Transporte activo
    # Active Transport
    'PAQ635': 'Active_Transport',
    'PAQ640': 'Active_Transport_Days',
    'PAD645': 'Active_Transport_Minutes',

    # Actividad física recreativa
    # Recreational Physical Activity
    'PAQ650': 'Vigorous_Recreation',
    'PAQ655': 'Vigorous_Recreation_Days',
    'PAD660': 'Vigorous_Recreation_Minutes',

    'PAQ665': 'Moderate_Recreation',
    'PAQ670': 'Moderate_Recreation_Days',
    'PAD675': 'Moderate_Recreation_Minutes',

    #Sedentarismo
    #Sedentary Lifestyle
    'PAD680': 'Sedentary_Minutes_Per_Day'
})

# ===========================
# Select variables
# ===========================

df_paq = df_paq[[
    "SEQN",

    "Vigorous_Work",
    "Vigorous_Work_Days",
    "Vigorous_Work_Minutes",

    "Moderate_Work",
    "Moderate_Work_Days",
    "Moderate_Work_Minutes",

    "Active_Transport",
    "Active_Transport_Days",
    "Active_Transport_Minutes",

    "Vigorous_Recreation",
    "Vigorous_Recreation_Days",
    "Vigorous_Recreation_Minutes",

    "Moderate_Recreation",
    "Moderate_Recreation_Days",
    "Moderate_Recreation_Minutes",

    "Sedentary_Minutes_Per_Day"
]].copy()


# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

categorical_columns = [
    "Vigorous_Work",
    "Moderate_Work",
    "Active_Transport",
    "Vigorous_Recreation",
    "Moderate_Recreation"
]

days_columns = [
    "Vigorous_Work_Days",
    "Moderate_Work_Days",
    "Active_Transport_Days",
    "Vigorous_Recreation_Days",
    "Moderate_Recreation_Days"
]

minutes_columns = [
    "Vigorous_Work_Minutes",
    "Moderate_Work_Minutes",
    "Active_Transport_Minutes",
    "Vigorous_Recreation_Minutes",
    "Moderate_Recreation_Minutes",
    "Sedentary_Minutes_Per_Day"
]

df_paq[categorical_columns] = df_paq[categorical_columns].replace(9, np.nan)
df_paq[days_columns] = df_paq[days_columns].replace(99, np.nan)
df_paq[minutes_columns] = df_paq[minutes_columns].replace(9999, np.nan)

df_paq["Sedentary_Minutes_Per_Day"] = (df_paq["Sedentary_Minutes_Per_Day"].replace(5.397605346934028e-79, 0))

# ===========================
# Replace coded values
# ===========================

activity_labels = {
    1: "Yes",
    2: "No"
}

df_paq[categorical_columns] = (
    df_paq[categorical_columns]
    .replace(activity_labels)
)

df_paq.head()


In [ ]:
#9 LIMPIEZA ARCHIVO Smoking - Cigarette Use (SMQ_J)
import pandas as pd
import numpy as np

# ===========================
# Load dataset
# ===========================

smq=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\SMQ_J.xpt")


smq = smq.rename(columns={
    'SMQ020': 'Ever_Smoked_100_Cigarettes',
    'SMD030': 'Smoking_Start_Age',
    'SMQ040': 'Current_Smoking_Status',
    'SMQ050Q': 'Time_Since_Quitting',
    'SMQ050U': 'Time_Since_Quitting_Unit',
    'SMD057': 'Cigarettes_Per_Day_When_Quit'
})

df_smoking = smq[[
    'SEQN',
    'Ever_Smoked_100_Cigarettes',
    'Smoking_Start_Age',
    'Current_Smoking_Status',
    'Time_Since_Quitting',
    'Time_Since_Quitting_Unit'
]].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================
categorical_columns = [
    "Ever_Smoked_100_Cigarettes",
    "Current_Smoking_Status",
    "Time_Since_Quitting_Unit"
]

df_smoking[categorical_columns] = df_smoking[categorical_columns].replace({
    7: np.nan,
    9: np.nan
})

# read_sas() occasionally imports some SAS zero values as 5.397605346934028e-79
df_smoking["Smoking_Start_Age"] = (
    df_smoking["Smoking_Start_Age"]
    .replace(5.397605346934028e-79, 0)
)

numeric_columns = [
    "Smoking_Start_Age",
    "Time_Since_Quitting"
]

df_smoking[numeric_columns] = df_smoking[numeric_columns].replace({
    777: np.nan,
    999: np.nan
})


# ===========================
# Replace coded values with descriptive labels
# ===========================

# Ever smoked at least 100 cigarettes
df_smoking["Ever_Smoked_100_Cigarettes"] = (
    df_smoking["Ever_Smoked_100_Cigarettes"]
    .replace({
        1: "Yes",
        2: "No"
    })
)

# Current smoking status
df_smoking["Current_Smoking_Status"] = (
    df_smoking["Current_Smoking_Status"]
    .replace({
        1: "Every day",
        2: "Some days",
        3: "Not at all"
    })
)

# Time since quitting unit
df_smoking["Time_Since_Quitting_Unit"] = (
    df_smoking["Time_Since_Quitting_Unit"]
    .replace({
        1: "Days",
        2: "Weeks",
        3: "Months",
        4: "Years"
    })
)

df_smoking.head()

In [ ]:
#10 CARGA Y LIMPIEZA ARCHIVO Alcohol Use (ALQ_J)
import pandas as pd
import numpy as np

df_alcohol=pd.read_sas(r"C:\Users\aleja\Documents\Analisis de Datos\Especialidad IT ACADEMY\Proyecto\Sueño\NHANES DATASETS\2017-2018 Demographics Data\ALQ_J.xpt")

# ===========================
# Rename variables
# ===========================

df_alcohol = df_alcohol.rename(columns={
    "ALQ111": "Ever_Had_Alcohol",
    "ALQ121": "Alcohol_Frequency_Last_12_Months",
    "ALQ130": "Average_Drinks_Per_Day",
    "ALQ142": "Heavy_Drinking_Frequency_Last_12_Months",
    "ALQ270": "Binge_Drinking_2h_Frequency_Last_12_Months",
    "ALQ280": "Days_With_8_Or_More_Drinks_Last_12_Months",
    "ALQ290": "Days_With_12_Or_More_Drinks_Last_12_Months",
    "ALQ151": "Ever_Been_Heavy_Drinker",
    "ALQ170": "Binge_Drinking_Last_30_Days"
})

# ===========================
# Select variables
# ===========================
df_alcohol = df_alcohol[[
    "SEQN",
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months",
    "Average_Drinks_Per_Day", # Value 15 represents "15 or more drinks"
    "Binge_Drinking_Last_30_Days"
]].copy()

# ===========================
# Replace NHANES missing value codes with NaN
# ===========================

categorical_columns = [
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months"
]

df_alcohol[categorical_columns] = df_alcohol[categorical_columns].replace({
    7: np.nan,
    9: np.nan,
    77: np.nan,
    99: np.nan
})

numeric_columns = [
    "Average_Drinks_Per_Day",
    "Binge_Drinking_Last_30_Days"
]

df_alcohol[numeric_columns] = df_alcohol[numeric_columns].replace({
    777: np.nan,
    999: np.nan
})

# ===========================
# Correct incorrectly imported SAS values
# ===========================

# read_sas() imports some 0 values as 5.397605346934028e-79

columns_with_zero_issue = [
    "Alcohol_Frequency_Last_12_Months",
    "Binge_Drinking_Last_30_Days"
]

for col in columns_with_zero_issue:
    df_alcohol[col] = df_alcohol[col].replace(
        5.397605346934028e-79,
        0
    )

# ===========================
# Replace coded values with descriptive labels
# ===========================

df_alcohol["Ever_Had_Alcohol"] = (
    df_alcohol["Ever_Had_Alcohol"]
    .replace({
        1: "Yes",
        2: "No"
    })
)

alcohol_frequency_labels = {
    0: "Never in the last 12 months",
    1: "Every day",
    2: "Nearly every day",
    3: "3-4 times per week",
    4: "Twice a week",
    5: "Once a week",
    6: "2-3 times per month",
    8: "Once a month",
    10: "3-11 times in the last year"
}

df_alcohol["Alcohol_Frequency_Last_12_Months"] = (
    df_alcohol["Alcohol_Frequency_Last_12_Months"]
    .replace(alcohol_frequency_labels)
)

df_alcohol.head()



# Realizando el Merge con los conjuntos de datos de NHANES

Los conjuntos de datos individuales de NHANES se fusionan en un único conjunto de datos analítico utilizando el identificador de participante (`SEQN`).

- Se utiliza `df_demo` como conjunto de datos base, ya que contiene la información demográfica y a todos los participantes incluidos en el estudio.
- Se aplica secuencialmente una **unión a la izquierda con un left join** a cada uno de los conjuntos de datos restantes para conservar a todos los participantes del conjunto de datos demográficos, incluso si falta información en algunos de los otros archivos.
- Tras cada fusión, se muestran las dimensiones del marco de datos resultante para verificar que el proceso de integración se ha realizado correctamente.

In [ ]:
# Dictionary containing all processed NHANES datasets
dfs = {
    'demo': df_demo,
    'sleep': df_sleep,
    'smoking': df_smoking,
    'alcohol': df_alcohol,
    'body': df_body,
    'physical_activity': df_paq,
    'Glycohemoglobin': df_ghb,
    'Blood Pressure': df_pressure,
    'Cholesterol': df_hdl
}

# Create a copy of the demographic dataset to use as the base dataframe
df_final = dfs['demo'].copy()

# Sequentially merge each dataset using the participant identifier (SEQN)
# A left join is used to retain all participants from the demographic dataset
for dataset_name, df_current in dfs.items():

    if dataset_name != 'demo':

        df_final = df_final.merge(
            df_current,
            on='SEQN',
            how='left'
        )

        # Display the dataframe shape after each merge
        print(f'{dataset_name}: {df_final.shape}')

## Comprobación de participantes duplicados

Antes de fusionar los conjuntos de datos, es importante verificar que el identificador del participante (`SEQN`) sea único dentro de cada marco de datos.

Dado que `SEQN` es la clave utilizada para fusionar todos los archivos de NHANES, los identificadores duplicados podrían dar lugar a fusiones incorrectas al generar filas duplicadas o relaciones «muchos a muchos». Esta validación garantiza que cada participante aparezca solo una vez en cada conjunto de datos antes del proceso de integración.

In [ ]:
# Check whether the participant identifier (SEQN) is unique in each dataset
for dataset_name, current_df in dfs.items():

    # Count duplicated participant IDs
    duplicated_ids = current_df['SEQN'].duplicated().sum()

    # Display the number of duplicates found
    print(dataset_name, duplicated_ids)

## Validación del conjunto de datos Obtenido

Se revisa el marco de datos fusionado para verificar su integridad antes de proceder a la limpieza de datos.

Se muestra la siguiente información:

- Número de filas y columnas.
- Número de identificadores de participantes duplicados (`SEQN`).
- Porcentaje de valores perdidos para cada variable.

In [ ]:
print("=" * 40)
print("DATAFRAME")
print("=" * 40)

# Display the dimensions of the merged dataset
print(f"Rows: {df_final.shape[0]}")
print(f"Columns: {df_final.shape[1]}")

# Verify that each participant appears only once
print(f"\nDuplicated SEQN values: {df_final['SEQN'].duplicated().sum()}")


## Evaluación de valores perdidos

Se genera una tabla resumen de los valores perdidos para evaluar la integridad de los datos antes de su limpieza.

Para cada variable, se muestra la siguiente información:

- Número de valores perdidos.
- Porcentaje de valores perdidos.
- Tipo de datos.

Solo se muestran las variables que contienen valores perdidos, ordenadas de mayor a menor según el porcentaje de datos perdidos.

In [ ]:
# Summarize missing values for each variable
na = pd.DataFrame({
    "NaN": df_final.isna().sum(),
    "%": (df_final.isna().mean() * 100).round(2),
    "Data Type": df_final.dtypes
})

# Keep only variables with missing values and sort them
na = (
    na[na["NaN"] > 0]
      .sort_values("%", ascending=False)
)

na

## Creación del conjunto de datos Específico para el analisis

La encuesta NHANES incluye a participantes de todas las edades; sin embargo, este estudio se centra exclusivamente en adultos de 20 años o más.

Por lo tanto, se creó un conjunto de datos analítico (`df_analysis`) seleccionando a los participantes adultos del conjunto de datos final.

Esta restricción se aplicó porque varias variables analizadas en este proyecto —entre ellas, la clasificación del IMC, el consumo de alcohol, el hábito tabáquico, la presión arterial y los marcadores bioquímicos— se definen de forma diferente o no están disponibles de manera sistemática para niños y adolescentes.

Limitar el análisis a los adultos mejora la coherencia metodológica y garantiza que todas las variables sean directamente comparables entre los participantes.

In [ ]:
# Create the analytical dataset
df_analysis = df_final.copy()

print(f"Participants before filtering: {len(df_analysis)}")

# Keep only participants aged 20 years or older
df_analysis = df_analysis[df_analysis["Age"] >= 20].copy()

print(f"Participants after filtering: {len(df_analysis)}")

# Reset the index after filtering
df_analysis.reset_index(drop=True, inplace=True)

## Frecuencia o Habitualidad del Tabaco

El cuestionario original sobre tabaquismo de la NHANES contiene varias variables que describen los antecedentes de tabaquismo (por ejemplo, tabaquismo a lo largo de la vida y hábitos tabáquicos actuales).

Para simplificar el análisis, estas variables se combinan en una única variable categórica (`Smoking_Status`).

Las categorías resultantes son:

- **Nunca ha fumado**
- **Exfumador**
- **Fumador actual**

Los participantes sobre los que no se dispone de información suficiente para determinar su situación respecto al tabaquismo se mantienen como datos faltantes (`NaN`).

In [ ]:
# Create an empty column to store the derived smoking status
df_analysis["Smoking_Status"] = pd.Series(dtype="object")

# Participants who have never smoked at least 100 cigarettes
df_analysis.loc[
    df_analysis["Ever_Smoked_100_Cigarettes"] == "No",
    "Smoking_Status"
] = "Never smoker"

# Participants who smoked in the past but do not currently smoke
df_analysis.loc[
    df_analysis["Current_Smoking_Status"] == "Not at all",
    "Smoking_Status"
] = "Former smoker"

# Participants who currently smoke
df_analysis.loc[
    df_analysis["Current_Smoking_Status"].isin(["Every day", "Some days"]),
    "Smoking_Status"
] = "Current smoker"


# Remove the original smoking variables after creating the summary variable
df_analysis.drop(
    columns=[
        "Ever_Smoked_100_Cigarettes",
        "Current_Smoking_Status",
        "Smoking_Start_Age",
        "Time_Since_Quitting",
        "Time_Since_Quitting_Unit"
    ],
    inplace=True,
    errors="ignore"
)


### Validación de la variable derivada

Se revisa la distribución de las categorías de la nueva variable `Smoking_Status` para verificar que la recodificación se ha realizado correctamente e identificar la proporción de valores perdidos.

In [ ]:
# Frequency distribution
df_analysis["Smoking_Status"].value_counts(dropna=False)

# Percentage distribution
round(
    df_analysis["Smoking_Status"]
        .value_counts(dropna=False, normalize=True) * 100,
    2
)

## Derivación de la variable `Alcohol_Consumption`

El cuestionario original de consumo de alcohol de NHANES incluye varias variables relacionadas con los hábitos de consumo, como el consumo de alcohol a lo largo de la vida y la frecuencia de consumo durante los últimos 12 meses.

Para este estudio, estas variables se combinaron en una única variable ordinal (`Alcohol_Consumption`) que representa el patrón habitual de consumo de alcohol de cada participante.

La clasificación conserva las categorías originales de respuesta de NHANES y proporciona una variable más sencilla para el análisis estadístico y la visualización de los datos.

| Alcohol_Consumption | Categoría NHANES |
|---------------------|------------------|
| Never | Nunca consumió alcohol |
| Former | No consumió alcohol durante los últimos 12 meses |
| Rare | 3–11 veces durante el último año |
| Occasional | Una vez al mes / 2–3 veces al mes |
| Regular | Una vez por semana / Dos veces por semana |
| Frequent | 3–4 veces por semana |
| Daily | Casi todos los días / Todos los días |

In [ ]:
# =====================================================
# Alcohol Consumption Category
# =====================================================

df_analysis["Alcohol_Consumption"] = pd.Series(dtype="object")

# Participants who have never consumed alcohol
df_analysis.loc[
    df_analysis["Ever_Had_Alcohol"] == "No",
    "Alcohol_Consumption"
] = "Never"

# Former drinkers (consumed alcohol in the past but not during the last year)
df_analysis.loc[
    (df_analysis["Ever_Had_Alcohol"] == "Yes") &
    (df_analysis["Alcohol_Frequency_Last_12_Months"] == "Never in the last 12 months"),
    "Alcohol_Consumption"
] = "Former"

# Rare drinkers
df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"] == "3-11 times in the last year",
    "Alcohol_Consumption"
] = "Rare"

# Monthly drinkers

df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Once a month",
        "2-3 times per month"
    ]),
    "Alcohol_Consumption"
] = "Occasional"

# Weekly drinkers
df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Once a week",
        "Twice a week"
    ]),
    "Alcohol_Consumption"
] = "Regular"

# Frequent weekly drinkers
df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"] == "3-4 times per week",
    "Alcohol_Consumption"
] = "Frequent"

# Daily drinkers
df_analysis.loc[
    df_analysis["Alcohol_Frequency_Last_12_Months"].isin([
        "Nearly every day",
        "Every day"
    ]),
    "Alcohol_Consumption"
] = "Daily"

# ==========================================================
# Create Alcohol_Consumption categorical variable
# ==========================================================

# Define the natural order of the categories.
# This ordering will be preserved in tables, plots and statistical analyses.
order = [
    "Never",
    "Former",
    "Rare",
    "Occasional",
    "Regular",
    "Frequent",
    "Daily"
]

df_analysis["Alcohol_Consumption"] = pd.Categorical(
    df_analysis["Alcohol_Consumption"],
    categories=order,
    ordered=True
)


df_alcohol.head()

## Validación de la variable `Alcohol_Consumption`

### Objetivo

Verificar que la variable derivada clasifica correctamente a los participantes según su patrón de consumo de alcohol.

### Validación

- Los participantes que respondieron **"No"** a `Ever_Had_Alcohol` se clasificaron como **Never**.
- Los participantes que respondieron **"Sí"** se clasificaron según la frecuencia de consumo de alcohol durante los últimos 12 meses.
- Los participantes con información insuficiente o ausente en el cuestionario original de NHANES conservaron valores perdidos (`NaN`).

### Resultado

La tabla de contingencia confirmó que todos los participantes fueron clasificados correctamente y que no se detectaron inconsistencias en el proceso de recodificación.

In [ ]:
pd.crosstab(
    df_analysis["Ever_Had_Alcohol"],
    df_analysis["Alcohol_Consumption"],
    dropna=False
)

## Derivación de la variable `BMI_Category`

El Índice de Masa Corporal (IMC) se transformó en una variable categórica ordinal siguiendo la clasificación establecida por la Organización Mundial de la Salud (OMS).

Esta transformación facilita la interpretación del estado nutricional de los participantes y permite realizar comparaciones entre grupos de IMC con relevancia clínica durante los análisis exploratorios y estadísticos.

In [ ]:
# ==========================================================
# Create BMI_Category according to WHO classification
# ==========================================================

# Initialize the new variable
df_analysis["BMI_Category"] = pd.Series(dtype="object")

# Underweight
df_analysis.loc[
    df_analysis["BMI"] < 18.5,
    "BMI_Category"
] = "Underweight"

# Normal weight
df_analysis.loc[
    (df_analysis["BMI"] >= 18.5) &
    (df_analysis["BMI"] < 25),
    "BMI_Category"
] = "Normal"

# Overweight
df_analysis.loc[
    (df_analysis["BMI"] >= 25) &
    (df_analysis["BMI"] < 30),
    "BMI_Category"
] = "Overweight"

# Obesity
df_analysis.loc[
    df_analysis["BMI"] >= 30,
    "BMI_Category"
] = "Obese"


# Define the natural order of BMI categories

bmi_order = [
    "Underweight",
    "Normal",
    "Overweight",
    "Obese"
]

df_analysis["BMI_Category"] = pd.Categorical(
    df_analysis["BMI_Category"],
    categories=bmi_order,
    ordered=True
)



## Derivación de la variable `Physical_Activity_Equivalent_Minutes_Week`

El Cuestionario de Actividad Física de NHANES recoge información sobre la actividad física realizada en distintos ámbitos, incluyendo la actividad ocupacional, el transporte activo y la actividad física recreativa.

Para este estudio, únicamente se consideraron la **actividad física recreativa** y el **transporte activo** para construir la variable de actividad física. La actividad física ocupacional se excluyó deliberadamente, ya que el objetivo del proyecto es evaluar los hábitos de vida de los participantes y no las exigencias físicas derivadas de su actividad laboral.

Siguiendo las recomendaciones de la Organización Mundial de la Salud (OMS), la actividad física recreativa vigorosa se ponderó con el doble de peso que la actividad moderada para calcular los **minutos equivalentes semanales**, reflejando su mayor intensidad y gasto energético. El transporte activo se consideró una actividad física de intensidad moderada.

La variable resultante (`Physical_Activity_Equivalent_Minutes_Week`) representa una estimación de los minutos equivalentes semanales de actividad física relacionada con el estilo de vida de cada participante.

In [ ]:
# ==========================================================
# Calculate weekly lifestyle-related physical activity
# ==========================================================

# Weekly minutes of moderate recreational physical activity
moderate_recreation = (
    df_analysis["Moderate_Recreation_Days"] *
    df_analysis["Moderate_Recreation_Minutes"]
)

# Weekly minutes of vigorous recreational physical activity
vigorous_recreation = (
    df_analysis["Vigorous_Recreation_Days"] *
    df_analysis["Vigorous_Recreation_Minutes"]
)

# Weekly minutes of active transportation
active_transport = (
    df_analysis["Active_Transport_Days"] *
    df_analysis["Active_Transport_Minutes"]
)

# Total weekly equivalent minutes of lifestyle-related physical activity
# Vigorous recreational activity is weighted twice according to WHO recommendations.
df_analysis["Physical_Activity_Equivalent_Minutes_Week"] = (
    moderate_recreation.fillna(0)
    + active_transport.fillna(0)
    + 2 * vigorous_recreation.fillna(0)
)


## Derivación de la variable `Physical_Activity_Level`

Los minutos equivalentes semanales de actividad física relacionada con el estilo de vida se transformaron en una variable categórica siguiendo las recomendaciones de la Organización Mundial de la Salud (OMS) para la actividad física en adultos.

Los participantes se clasificaron en cuatro categorías según sus minutos equivalentes semanales de actividad física de intensidad moderada:

- **Inactive:** 0 minutos/semana
- **Below Recommendation:** 1–149 minutos/semana
- **Meets Recommendation:** 150–299 minutos/semana
- **Exceeds Recommendation:** ≥300 minutos/semana

Estas categorías indican si los participantes cumplen o superan los niveles mínimos de actividad física recomendados por la OMS y no deben interpretarse como una medida absoluta de condición física.

In [ ]:
# ==========================================================
# Create Physical Activity Level
# ==========================================================

# Categorize participants according to the WHO physical
# activity recommendations
df_analysis["Physical_Activity_Level"] = pd.cut(
    df_analysis["Physical_Activity_Equivalent_Minutes_Week"],
    bins=[-1, 0, 149, 299, float("inf")],
    labels=[
        "Inactive",
        "Below Recommendation",
        "Meets Recommendation",
        "Exceeds Recommendation"
    ]
)

# Define the logical order of the categories
activity_order = [
    "Inactive",
    "Below Recommendation",
    "Meets Recommendation",
    "Exceeds Recommendation"
]

df_analysis["Physical_Activity_Level"] = pd.Categorical(
    df_analysis["Physical_Activity_Level"],
    categories=activity_order,
    ordered=True
)

# Display the distribution
display(df_analysis["Physical_Activity_Level"].value_counts(dropna=False))

## Derivación de la variable `Average_Sleep_Hours`

El Cuestionario de Trastornos del Sueño de NHANES registra la duración habitual del sueño de los participantes por separado para los días laborables y los fines de semana.

En lugar de analizar ambos valores de forma independiente, se creó una nueva variable (`Average_Sleep_Hours`) que representa la **duración media semanal del sueño**. Esta variable proporciona una estimación más representativa del patrón habitual de sueño al tener en cuenta el diferente número de días laborables y de fin de semana que componen una semana típica.

La duración media semanal del sueño se calculó mediante una media ponderada considerando:

- **5 días laborables**
- **2 días de fin de semana**

Este procedimiento refleja con mayor precisión el patrón global de sueño de los participantes que una media aritmética simple.

In [ ]:
# ==========================================================
# Calculate Average Weekly Sleep Duration
# ==========================================================

# Calculate the weighted average weekly sleep duration.
# A typical week consists of five workdays and two weekend days.
# Therefore, workday sleep duration receives a weight of 5,
# while weekend sleep duration receives a weight of 2.

df_analysis["Average_Sleep_Hours"] = (
    (
        df_analysis["Sleep_Hours_Workdays"] * 5
        + df_analysis["Sleep_Hours_Weekend"] * 2
    ) / 7
)


## Derivación de la variable `Sleep_Duration_Category`

Con el fin de facilitar la interpretación de los resultados, la variable `Average_Sleep_Hours` se transformó en una variable categórica siguiendo las recomendaciones de la **American Academy of Sleep Medicine (AASM)** y la **Sleep Research Society** para adultos sanos.

Los participantes se clasificaron en tres categorías según su duración media semanal del sueño:

- **Short Sleep:** menos de 7 horas por noche.
- **Recommended Sleep:** entre 7 y 9 horas por noche.
- **Long Sleep:** más de 9 horas por noche.

Estas categorías representan los rangos de duración del sueño recomendados para la población adulta y se utilizan habitualmente en estudios epidemiológicos que investigan la relación entre la duración del sueño y distintos indicadores de salud.

In [ ]:
# ==========================================================
# Create Sleep Duration Category
# ==========================================================

# Create the new column
df_analysis["Sleep_Duration_Category"] = pd.Series(dtype='object')

# Less than 7 hours
df_analysis.loc[
    df_analysis["Average_Sleep_Hours"] < 7,
    "Sleep_Duration_Category"
] = "Short Sleep"

# Between 7 and 9 hours (inclusive)
df_analysis.loc[
    (df_analysis["Average_Sleep_Hours"] >= 7) &
    (df_analysis["Average_Sleep_Hours"] <= 9),
    "Sleep_Duration_Category"
] = "Recommended Sleep"

# More than 9 hours
df_analysis.loc[
    df_analysis["Average_Sleep_Hours"] > 9,
    "Sleep_Duration_Category"
] = "Long Sleep"

# Ordered categorical variable
sleep_order = [
    "Short Sleep",
    "Recommended Sleep",
    "Long Sleep"
]

df_analysis["Sleep_Duration_Category"] = pd.Categorical(
    df_analysis["Sleep_Duration_Category"],
    categories=sleep_order,
    ordered=True
)


## Simplificación de las variables de actividad física

Una vez creadas las variables derivadas de actividad física, varias variables originales del cuestionario de NHANES se eliminan del conjunto de datos analítico, ya que dejan de aportar información adicional para los objetivos de este estudio.

Las variables originales describen la frecuencia (días) y la duración (minutos) de distintos tipos de actividad física, incluyendo la actividad ocupacional, el transporte activo y la actividad física recreativa. Esta información se ha consolidado en las siguientes variables derivadas:

- **Physical_Activity_Equivalent_Minutes_Week**: representa los minutos equivalentes semanales de actividad física relacionada con el estilo de vida, calculados a partir de la actividad física recreativa y el transporte activo siguiendo las recomendaciones de la OMS.
- **Physical_Activity_Level**: clasifica a los participantes según las recomendaciones de actividad física de la Organización Mundial de la Salud (OMS).

Por ello, las variables originales relacionadas con la actividad física se eliminan para reducir la complejidad del conjunto de datos y evitar información redundante. No obstante, **Sedentary_Minutes_Per_Day** se conserva como una variable continua, ya que puede aportar información adicional en los análisis estadísticos y las visualizaciones.

Esta simplificación mejora la legibilidad del conjunto de datos, facilita el análisis exploratorio de los datos (EDA) y reduce el número de variables sin perder información relevante.

In [ ]:
# ==========================================================
# Remove original physical activity variables
# ==========================================================

columns_to_drop = [
    "Moderate_Work_Days",
    "Moderate_Work_Minutes",
    "Vigorous_Work_Days",
    "Vigorous_Work_Minutes",
    "Moderate_Recreation_Days",
    "Moderate_Recreation_Minutes",
    "Vigorous_Recreation_Days",
    "Vigorous_Recreation_Minutes",
    "Active_Transport_Days",
    "Active_Transport_Minutes"
]

df_analysis.drop(columns=columns_to_drop, inplace=True,errors='ignore')
print(f"{len(columns_to_drop)} original alcohol variables were removed.")



## Simplificación final de las variables de actividad física

Tras crear las variables derivadas de actividad física, las variables binarias originales que indicaban si los participantes realizaban determinados tipos de actividad física (ocupacional, transporte activo o actividad física recreativa) dejaron de ser necesarias para los análisis previstos.

El estudio utiliza las siguientes variables para representar el nivel de actividad física de los participantes:

- **Physical_Activity_Equivalent_Minutes_Week**: representa los minutos equivalentes semanales de actividad física, calculados siguiendo las recomendaciones de la Organización Mundial de la Salud (OMS), ponderando la actividad física recreativa vigorosa con el doble de peso que la actividad moderada.
- **Physical_Activity_Level**: clasifica a los participantes según las recomendaciones de actividad física de la Organización Mundial de la Salud (OMS).
- **Sedentary_Minutes_Per_Day**: medida continua del tiempo diario dedicado a actividades sedentarias.

Dado que estas variables resumen la información relevante sobre la actividad física de los participantes, las variables binarias originales se eliminaron para simplificar aún más el conjunto de datos y evitar información redundante.

Esta simplificación final da lugar a un conjunto de datos más compacto y manejable, manteniendo toda la información necesaria para responder a los objetivos e hipótesis del estudio.

In [ ]:
# ==========================================================
# 14. FINAL PHYSICAL ACTIVITY VARIABLES SIMPLIFICATION
# ==========================================================
# After creating the summary physical activity variables,
# the original binary indicators describing whether
# participants performed each type of activity became
# redundant for the objectives of this study.
#
# The analysis will use:
# - Physical_Activity_Equivalent_Minutes_Week
# - Physical_Activity_Level
# - Sedentary_Minutes_Per_Day
#
# These variables summarize the participants' overall
# physical activity while keeping the dataset more concise.
# ==========================================================

# Original binary physical activity variables to remove
physical_activity_binary_columns = [
    "Vigorous_Work",
    "Moderate_Work",
    "Active_Transport",
    "Vigorous_Recreation",
    "Moderate_Recreation"
    
]

# Remove redundant variables
df_analysis.drop(columns=physical_activity_binary_columns, inplace=True,errors='ignore')

print(f"{len(physical_activity_binary_columns)} binary physical activity variables were removed.")


## Simplificación de las variables de consumo de alcohol

Una vez creada y validada la variable derivada `Alcohol_Consumption`, las variables originales del cuestionario de consumo de alcohol de NHANES se eliminan del conjunto de datos analítico, ya que dejan de aportar información adicional para los objetivos de este estudio.

Las variables originales describen distintos aspectos del consumo de alcohol, como el consumo a lo largo de la vida, la frecuencia de consumo, la cantidad media de alcohol ingerida y los episodios de consumo intensivo (*binge drinking*). Esta información se ha consolidado en la variable derivada `Alcohol_Consumption`, que resume el patrón habitual de consumo de alcohol de los participantes en categorías fácilmente interpretables.

Por ello, las variables originales del cuestionario de alcohol se eliminan para reducir la complejidad del conjunto de datos y evitar información redundante.

Se eliminan las siguientes variables:

- **Ever_Had_Alcohol**
- **Alcohol_Frequency_Last_12_Months**
- **Average_Drinks_Per_Day**
- **Binge_Drinking_Last_30_Days**

Esta simplificación mejora la legibilidad del conjunto de datos y facilita los análisis estadísticos y la visualización de los datos, manteniendo toda la información necesaria para el estudio.

In [ ]:
# ==========================================================
# 13. ALCOHOL VARIABLES SIMPLIFICATION
# ==========================================================
# After creating the derived variable Alcohol_Consumption,
# the original alcohol questionnaire variables became
# redundant and were removed.
#
# Alcohol_Consumption summarizes participants' drinking
# patterns into meaningful categories that will be used
# throughout the analysis.
# ==========================================================

# Original alcohol variables to remove
alcohol_columns = [
    "Ever_Had_Alcohol",
    "Alcohol_Frequency_Last_12_Months",
    "Average_Drinks_Per_Day",
    "Binge_Drinking_Last_30_Days"
]

# Remove redundant variables
df_analysis.drop(columns=alcohol_columns, inplace=True,errors='ignore')

print(f"{len(alcohol_columns)} original alcohol variables were removed.")

## Simplificación de las variables de sueño

Una vez creadas las variables derivadas `Average_Sleep_Hours` y `Sleep_Duration_Category`, las variables originales de NHANES que describen la hora de acostarse y la hora de despertarse se eliminan del conjunto de datos analítico, ya que dejan de aportar información adicional para los objetivos de este estudio.

El objetivo principal de este proyecto es investigar la relación entre la duración del sueño y distintos indicadores de salud, más que analizar los horarios habituales de sueño. La información proporcionada por las variables originales relacionadas con los horarios de sueño ha quedado resumida en la variable derivada `Average_Sleep_Hours`.

No obstante, las variables `Sleep_Hours_Workdays` y `Sleep_Hours_Weekend` se conservan, ya que permiten comparar la duración del sueño entre los días laborables y los fines de semana, lo que puede aportar información adicional durante el análisis exploratorio de los datos (EDA).

Se eliminan las siguientes variables:

- **Bedtime_Workdays**
- **Wakeup_Workdays**
- **Bedtime_Weekend**
- **Wakeup_Weekend**

Esta simplificación mejora la legibilidad del conjunto de datos, reduce su complejidad y conserva toda la información necesaria para responder a los objetivos del estudio.

In [ ]:
# ==========================================================
# 14. SLEEP VARIABLES SIMPLIFICATION
# ==========================================================
# After creating the derived variables Average_Sleep_Hours
# and Sleep_Duration_Category, the exact bedtime and wake-up
# time variables became unnecessary for the objectives of
# this study.
#
# Sleep_Hours_Workdays and Sleep_Hours_Weekend are retained
# because they allow comparisons between weekday and weekend
# sleep duration, which may provide additional insights
# during the exploratory data analysis (EDA).
# ==========================================================

# Original sleep schedule variables to remove
sleep_columns = [
    "Bedtime_Workdays",
    "Wakeup_Workdays",
    "Bedtime_Weekend",
    "Wakeup_Weekend"
]

# Remove redundant variables
df_analysis.drop(columns=sleep_columns, inplace=True,errors='ignore')

print(f"{len(sleep_columns)} original sleep schedule variables were removed.")

## Simplificación de las variables antropométricas

Una vez creada la variable derivada `BMI_Category`, se revisan las variables antropométricas para conservar únicamente aquellas que son directamente relevantes para los objetivos de este estudio.

`BMI` se mantiene como el principal indicador de adiposidad general, junto con `BMI_Category`, que facilita los análisis categóricos y la visualización de los datos. Asimismo, `Waist_Circumference` se conserva por ser un indicador importante de obesidad abdominal y aportar información complementaria al IMC.

Las variables **Weight**, **Height** y **Hip_Circumference** se eliminan. El peso y la talla se utilizaron principalmente para calcular el IMC, mientras que la circunferencia de la cadera no es necesaria para los análisis previstos, ya que la relación cintura-cadera (WHR) no forma parte de este estudio.

Esta simplificación mejora la legibilidad del conjunto de datos, reduce la redundancia y conserva las variables antropométricas necesarias para responder a los objetivos de la investigación.

In [ ]:
# ==========================================================
# 13. BODY MEASUREMENT VARIABLES SIMPLIFICATION
# ==========================================================
# Weight and Height were removed because BMI already combines
# both measurements into a single indicator of overall body
# adiposity.
#
# Hip_Circumference was also removed because it is mainly used
# to calculate the waist-to-hip ratio (WHR), which is not
# included in this study.
#
# BMI, BMI_Category and Waist_Circumference were retained as
# the main anthropometric variables for the analysis.
# ==========================================================

body_measurement_columns = [
    "Weight",
    "Height",
    "Hip_Circumference"
]

# Remove redundant variables
df_analysis.drop(columns=body_measurement_columns, inplace=True,errors='ignore')

print(f"{len(body_measurement_columns)} body measurement variables were removed.")


In [ ]:
#CREAMOS UN DATAFRAME NUEVO SIN LOS NULOS y luego rellenar el poverty index

#Rellenamos los valores nulos de Poverty_index 
df_analysis["Age_Group"] = pd.cut(
    df_analysis["Age"],
    bins=[20, 40, 60, 80],
    labels=["20-39", "40-59", "60-80"],
    include_lowest=True)

group_median = (df_analysis.groupby(["Gender", "Education", "Age_Group"],observed=False)["Poverty_Index"].transform("median"))

# Remove auxiliary variable
df_analysis.drop(columns="Age_Group", inplace=True)

df_analysis["Poverty_Index"] = (
    df_analysis["Poverty_Index"]
    .fillna(group_median))

df = df_analysis.dropna(
    subset=["Alcohol_Consumption"]
).copy()

print("Original:", df_analysis.shape)
print("Sin NaN en Alcohol_Consumption:", df.shape)

df.isna().sum().sort_values(ascending=False)
#df_analysis.isnull().sum().sort_values(ascending=False)

df = df.reset_index(drop=True)


# Análisis Exploratorio de los Datos (EDA)

El análisis exploratorio de los datos (EDA) comienza con una descripción de la población de estudio.

Antes de investigar la relación entre la duración del sueño y las variables relacionadas con la salud, es fundamental conocer las características demográficas de los participantes incluidos en el estudio.

En esta sección se presenta una visión general de la muestra mediante estadísticas descriptivas y representaciones gráficas de las principales variables demográficas.

## Población de estudio

La población de estudio se describe mediante variables demográficas tanto cuantitativas como cualitativas.

**Variables cuantitativas:**

- `Age`
- `Poverty_Index`

**Variables cualitativas:**

- `Gender`
- `Race`
- `Education`
- `Marital_Status`

Las variables cuantitativas se resumen mediante estadísticos descriptivos, mientras que las variables cualitativas se describen mediante distribuciones de frecuencias y porcentajes.

In [ ]:
# ==========================================================
# Study Population - Numerical Variables
# ==========================================================

# Impute missing Poverty Index values using the median within
# Gender × Education × Age Group strata to reduce missingness
# while preserving the demographic structure of the sample.


# Select numerical demographic variables
numerical_variables = ["Age", "Poverty_Index"]

# Summary statistics
summary_statistics = df[numerical_variables].describe().T

# Add missing values
summary_statistics["Missing"] = (
    df[numerical_variables]
    .isna()
    .sum()
)

# Reorder columns
summary_statistics = summary_statistics[
    ["count", "Missing", "mean", "std", "min", "25%", "50%", "75%", "max"]
]

display(summary_statistics.round(2))


### Interpretación

La población de estudio estuvo compuesta por **4.103 participantes adultos** de **20 años o más**.

La edad media de los participantes fue de **51,8 años** (DE = **17,7**), con una mediana de **54 años**, lo que indica que la muestra estuvo formada principalmente por adultos de mediana edad y adultos mayores.

Con el objetivo de reducir la cantidad de datos faltantes, los valores ausentes de la variable **Poverty_Index** fueron imputados utilizando la mediana calculada dentro de cada estrato definido por el **sexo, el nivel educativo y el grupo de edad**. Tras este procedimiento, el **Poverty_Index** estuvo disponible para **4.097 participantes**, permaneciendo únicamente **6 observaciones (0,15%)** con valores ausentes. El valor medio del **Poverty_Index** fue de **2,53** (DE = **1,57**), lo que refleja una considerable variabilidad socioeconómica dentro de la población estudiada.

> **Nota:** En NHANES, todos los participantes con **80 años o más** se registran como **80 años** para proteger la confidencialidad de los participantes.

## Variables cualitativas

Las principales variables demográficas cualitativas se resumieron mediante distribuciones de frecuencias y porcentajes con el fin de describir la composición de la población de estudio.

Las variables incluidas en esta sección son:

- `Gender`
- `Race`
- `Education`
- `Marital_Status`

Estas variables proporcionan una descripción general de la muestra de estudio y ofrecen el contexto necesario para los análisis posteriores.

Dado que el objetivo principal de este estudio es examinar la asociación entre la duración del sueño y las variables relacionadas con la salud, únicamente se presentan tablas descriptivas de frecuencias, sin visualizaciones gráficas adicionales.

In [ ]:
study_population_cat = df[[
    "Gender",
    "Race",
    "Education",
    "Marital_Status"
]]

for col in study_population_cat:
    print(f"\n{'='*60}")
    print(col)
    print('='*60)

    freq = (
        df[col]
        .value_counts(dropna=False)
        .rename("Count")
        .to_frame()
    )

    freq["Percentage"] = (
        freq["Count"] / len(df) * 100
    ).round(2)

    display(freq)
   

# Perfil del sueño

La duración del sueño constituye la principal variable de interés de este estudio y sirve como base para todos los análisis posteriores. Antes de examinar su asociación con las variables relacionadas con la salud, es fundamental conocer las características del sueño de la población de estudio.

En esta sección se presenta un análisis descriptivo de la duración del sueño mediante estadísticas descriptivas y representaciones gráficas. En concreto, se utilizan estadísticos descriptivos, un histograma y un diagrama de caja para evaluar la distribución, la tendencia central, la variabilidad y la posible presencia de valores atípicos de la variable `Average_Sleep_Hours`.

## Duración media del sueño

La variable `Average_Sleep_Hours` representa el número medio de horas de sueño por noche, calculado a partir de la duración del sueño autodeclarada por los participantes durante los días laborables y los fines de semana.

El análisis comienza con el examen de sus estadísticos descriptivos, seguido de representaciones gráficas que permiten comprender mejor la distribución de la duración del sueño en la población de estudio.

In [ ]:
sleep_stats = df["Average_Sleep_Hours"].describe().to_frame()

display(sleep_stats)

### Interpretación

La variable `Average_Sleep_Hours` estuvo disponible para **4.052 participantes**, mientras que **51 participantes (1,24 %)** presentaron valores perdidos.

La duración media del sueño fue de **7,75 horas por noche** (DE = **1,52**), con una mediana de **7,86 horas**, lo que indica que el centro de la distribución se sitúa próximo al intervalo de sueño recomendado para la población adulta.

El 50 % central de los participantes durmió entre **7,00 y 8,64 horas** por noche (RIC = **1,64 horas**), lo que refleja una variabilidad moderada en la duración del sueño dentro de la población de estudio.

La duración del sueño osciló entre **2,0 y 14,0 horas** por noche, lo que pone de manifiesto la presencia de participantes con duraciones de sueño muy cortas y muy largas. Estos valores se examinarán con mayor detalle mediante representaciones gráficas para evaluar la distribución de la variable e identificar posibles valores atípicos.

In [ ]:
#Distribucion de la media de horas de sueño con un Histograma
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x="Average_Sleep_Hours",
    bins=20,
    kde=True
)

plt.title("Distribution of Average Sleep Duration")
plt.xlabel("Average Sleep Hours")
plt.ylabel("Frequency")

plt.show()
print('Asimetría: ',df["Average_Sleep_Hours"].skew())

In [ ]:
#Distribucion del promedio de horas de sueño, Boxplot
plt.figure(figsize=(8,2.5))

sns.boxplot(
    x=df["Average_Sleep_Hours"]
)

plt.title("Boxplot of Average Sleep Duration")

plt.show()

In [ ]:
#Se muestran los rangos y el numero de outliers
Q1 = df["Average_Sleep_Hours"].quantile(0.25)
Q3 = df["Average_Sleep_Hours"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df["Average_Sleep_Hours"] < lower) |
    (df["Average_Sleep_Hours"] > upper)
]

print("Lower limit:", lower)
print("Upper limit:", upper)
print("Number of outliers:", len(outliers))

### Interpretación

El histograma mostró que la distribución de `Average_Sleep_Hours` era aproximadamente simétrica, lo que resulta coherente con la similitud entre la media (**7,79 horas**) y la mediana (**7,86 horas**). Además, el coeficiente de asimetría (**-0,03**) confirma que la distribución presenta una asimetría prácticamente nula.

El diagrama de caja identificó **224 posibles valores atípicos** (aproximadamente el **4 %** de la muestra), correspondientes a participantes que declararon duraciones medias de sueño muy cortas o muy largas. Dado que estos valores son plausibles dentro de la variabilidad natural del comportamiento del sueño y no existen indicios de errores de registro o medición, se conservaron para los análisis posteriores.

# Categorías de duración del sueño

Con el fin de facilitar la interpretación y la comparación de la duración del sueño, los participantes se clasificaron en tres categorías siguiendo las recomendaciones de la National Sleep Foundation:

- **Short Sleep:** menos de 7 horas por noche.
- **Recommended Sleep:** entre 7 y 9 horas por noche.
- **Long Sleep:** más de 9 horas por noche.

Esta variable categórica constituye la principal variable de agrupación utilizada en los análisis posteriores, permitiendo comparar las variables relacionadas con la salud y los factores del estilo de vida entre los distintos grupos de duración del sueño.

In [ ]:
# Frequency and percentage distribution of sleep duration categories

sleep_category = (
    df["Sleep_Duration_Category"]
    .value_counts(dropna=False)
    .rename("Count")
    .to_frame()
)

sleep_category["Percentage"] = (
    sleep_category["Count"] / len(df) * 100
).round(2)

print(sleep_category)

In [ ]:
#Se muestra la distribucion de los participantes por categoria de sueño 
import matplotlib.pyplot as plt
import seaborn as sns

# Define the order of the sleep categories
order = [
    "Short Sleep",
    "Recommended Sleep",
    "Long Sleep"
]

# Create the figure
plt.figure(figsize=(8, 5))

# Create the bar chart
ax = sns.countplot(
    data=df,
    x="Sleep_Duration_Category",
    order=order,
    color="steelblue"
)

# Add the number of participants above each bar
for container in ax.containers:
    ax.bar_label(container, fontsize=10)

# Customize the graph
plt.title("Distribution of Sleep Duration Categories", fontsize=14)
#plt.xlabel("Sleep Duration Category", fontsize=12)
plt.xlabel('')
plt.ylabel("Number of Participants", fontsize=12)

plt.tight_layout()
plt.show()

print(df.groupby('Sleep_Duration_Category').size())


### Interpretación

La mayoría de los participantes (**60,00 %**) indicó que dormía entre **7 y 9 horas por noche**, lo que se corresponde con la duración recomendada del sueño.

Aproximadamente el **24,66 %** de los participantes se clasificó como **«sueño corto»**, mientras que el **14,09 %** se clasificó como **«sueño largo»**. Solo el **1,24 %** de las observaciones presentó datos faltantes para esta variable.

En general, estos resultados indican que, aunque la mayoría de los participantes alcanzaba la duración de sueño recomendada, una proporción importante de la población del estudio declaró dormir menos o más de lo recomendado. Estas categorías de duración del sueño se utilizarán en los análisis posteriores para investigar su asociación con características demográficas, indicadores de salud y factores relacionados con el estilo de vida.

## Health Profile

Con el objetivo de obtener una evaluación más completa del estado general de salud de los participantes, se creó una variable compuesta denominada **Health Profile**, combinando cuatro indicadores clínicos ampliamente utilizados: el índice de masa corporal (IMC), la hemoglobina glicosilada (HbA1c), la presión arterial y el colesterol HDL.

Cada indicador fue clasificado siguiendo las recomendaciones de organismos internacionales como la Organización Mundial de la Salud (OMS), la American Diabetes Association (ADA) y la American Heart Association (AHA). Todos los indicadores contribuyen por igual a la evaluación del estado de salud, manteniendo su interpretación clínica.

A cada indicador se le asignó una puntuación de **2 puntos** cuando se encontraba en un rango saludable, **1 punto** cuando presentaba un riesgo intermedio (cuando correspondía) y **0 puntos** cuando se encontraba en un rango desfavorable.

Para reducir la pérdida de participantes debido a valores ausentes en algunos biomarcadores, el **Health Profile** no se calculó como una suma directa de puntos, sino como el **porcentaje de la puntuación máxima posible según los indicadores disponibles para cada participante**. De este modo, cada participante fue evaluado únicamente con la información clínica disponible. Aquellos con **menos de tres indicadores clínicos disponibles** no fueron clasificados.

| Indicador clínico | Guía | Saludable (2) | Intermedio (1) | Desfavorable (0) |
|-------------------|------|---------------|----------------|------------------|
| IMC | OMS | Peso normal | Bajo peso / Sobrepeso | Obesidad |
| HbA1c | ADA | Normal | Prediabetes | Diabetes |
| Presión arterial | AHA/ACC | Normal | Elevada | Hipertensión |
| Colesterol HDL | NHLBI / AHA | Normal | — | Bajo |

### Clasificación del Health Profile

| Porcentaje obtenido | Health Profile |
|---------------------:|----------------|
| ≥ 80% | Good Health Profile |
| 50–79,99% | Intermediate Health Profile |
| < 50% | Poor Health Profile |

> **Nota:** Los participantes con menos de tres indicadores clínicos disponibles fueron clasificados como valores perdidos (`missing`) para esta variable.

In [ ]:
# ==========================
# Health Profile - Scoring
# ==========================

# BMI (WHO)
BMI_Points = np.select([df["BMI_Category"] == "Normal", df["BMI_Category"].isin(["Underweight", "Overweight"])
                        ,df["BMI_Category"] == "Obese"], [2,1,0], default=np.nan)

# HbA1c (ADA)
HbA1c_Points = np.select(
    [
        df["HbA1c"] < 5.7,
        (df["HbA1c"] >= 5.7) & (df["HbA1c"] < 6.5),
        df["HbA1c"] >= 6.5
    ],
    [2, 1, 0],
    default=np.nan
)

# Blood Pressure (AHA/ACC)
BP_Points = np.select(
    [
        (df["Systolic_BP"] < 120) &
        (df["Diastolic_BP"] < 80),

        (df["Systolic_BP"].between(120, 129)) &
        (df["Diastolic_BP"] < 80),

        (df["Systolic_BP"] >= 130) |
        (df["Diastolic_BP"] >= 80)
    ],
    [2, 1, 0],
    default=np.nan
)

# HDL Cholesterol (NHLBI/AHA)
HDL_Points = np.select(
    [
        ((df["Gender"] == "Male") & (df["HDL"] >= 40)) |
        ((df["Gender"] == "Female") & (df["HDL"] >= 50)),

        ((df["Gender"] == "Male") & (df["HDL"] < 40)) |
        ((df["Gender"] == "Female") & (df["HDL"] < 50))
    ],
    [2, 0],
    default=np.nan
)


In [ ]:
#USANDO PORCENTAJES PARA EVITAR LOS NULOS POR NO TENER UNA DE LAS 4 VARIABLES 

# ==========================
# Health Profile Score
# ==========================

health_points = pd.DataFrame({
    "BMI": BMI_Points,
    "HbA1c": HbA1c_Points,
    "BP": BP_Points,
    "HDL": HDL_Points
})

# Sum of available points
obtained = health_points.sum(axis=1, skipna=True)

# Maximum possible score based on available variables
available = health_points.notna().sum(axis=1) * 2

# Percentage score (0–100)
df["Health_Profile_Percentage"] = (
    obtained / available * 100).round(2)

# Require at least 3 available indicators
df.loc[available < 6, "Health_Profile_Percentage"] = np.nan

# ==========================
# Health Profile Classification
# ==========================

df["Health_Profile"] = np.select(
    [
        df["Health_Profile_Percentage"] >= 80,
        df["Health_Profile_Percentage"].between(50, 79.99),
        df["Health_Profile_Percentage"] < 50
    ],
    [
        "Good Health Profile",
        "Intermediate Health Profile",
        "Poor Health Profile"
    ],
    default=None
)

display(df["Health_Profile"].value_counts(dropna=False))

## Interpretación

Un total de **753 participantes (18,35%)** fueron clasificados con un **Good Health Profile**, **1.964 (47,87%)** con un **Intermediate Health Profile** y **1.206 (29,39%)** con un **Poor Health Profile**.

Por otro lado, **180 participantes (4,39%)** no pudieron ser clasificados porque disponían de **menos de tres de los cuatro indicadores clínicos** utilizados para construir el *Health Profile* (`BMI`, `HbA1c`, `Blood Pressure` y `HDL Cholesterol`). Este criterio se estableció para garantizar que la clasificación estuviera basada en información clínica suficiente, al mismo tiempo que se minimizaba la pérdida de participantes debida a valores ausentes.

En conjunto, aproximadamente **el 95,6% de la población del estudio** pudo clasificarse en una de las tres categorías del *Health Profile*. Esto indica que la metodología adoptada permitió reducir la pérdida de datos sin comprometer la validez clínica de la variable compuesta. Además, la distribución observada refleja una variabilidad considerable en el estado general de salud de los participantes, proporcionando una base adecuada para explorar la asociación entre la duración del sueño y los indicadores de salud en los análisis posteriores.

## Lifestyle Profile

Con el objetivo de resumir los hábitos de vida de los participantes, se creó una variable compuesta denominada **Lifestyle Profile**, combinando tres comportamientos modificables relacionados con la salud: la actividad física, el hábito tabáquico y el consumo de alcohol.

La actividad física se clasificó siguiendo las recomendaciones de la Organización Mundial de la Salud (OMS), mientras que el hábito tabáquico y el consumo de alcohol se categorizaron utilizando variables derivadas de la National Health and Nutrition Examination Survey (NHANES) y las correspondientes directrices de los Centers for Disease Control and Prevention (CDC). Los tres indicadores contribuyeron por igual a la evaluación del estilo de vida de los participantes.

El tiempo sedentario se excluyó intencionadamente de esta variable compuesta. Aunque el comportamiento sedentario prolongado se reconoce como un importante factor de riesgo para la salud, las recomendaciones actuales de la OMS no establecen puntos de corte universalmente aceptados para clasificar el tiempo sedentario diario en categorías de riesgo. Para evitar introducir umbrales arbitrarios, esta variable se analizó posteriormente de forma independiente como una variable continua.

A cada indicador se le asignaron **2 puntos** cuando representaba el comportamiento más saludable, **1 punto** cuando correspondía a un comportamiento intermedio y **0 puntos** cuando representaba el comportamiento menos favorable.

Con el fin de reducir la pérdida de participantes debida a valores ausentes en alguno de los indicadores del estilo de vida, el **Lifestyle Profile** no se calculó como una suma directa de puntos, sino como el **porcentaje de la puntuación máxima posible según los indicadores disponibles para cada participante**. De esta forma, cada participante fue evaluado utilizando únicamente la información de estilo de vida disponible. Aquellos con **menos de dos de los tres indicadores de estilo de vida disponibles** no fueron clasificados.

| Indicador | Guía | Saludable (2) | Intermedio (1) | Desfavorable (0) |
|------------|------|---------------|----------------|------------------|
| Actividad física | OMS | Cumple / Supera la recomendación | Por debajo de la recomendación | Inactivo |
| Hábito tabáquico | CDC | Nunca fumador | Exfumador | Fumador actual |
| Consumo de alcohol | CDC (NHANES) | Nunca / Raro / Ocasional | Exconsumidor / Regular | Frecuente / Diario |

### Clasificación del Lifestyle Profile

| Porcentaje obtenido | Lifestyle Profile |
|---------------------:|-------------------|
| ≥ 80% | Healthy Lifestyle |
| 50–79,99% | Intermediate Lifestyle |
| < 50% | Unhealthy Lifestyle |

> **Nota:** Los participantes con menos de **dos de los tres indicadores de estilo de vida disponibles** fueron clasificados como valores perdidos (`missing`) para esta variable.

## Puntuación de la actividad física

La actividad física se puntuó siguiendo las recomendaciones de la Organización Mundial de la Salud (OMS).

Los participantes que cumplían o superaban los niveles recomendados de actividad física recibieron la puntuación más alta (**2 puntos**), aquellos que realizaban alguna actividad física, pero por debajo de las recomendaciones, recibieron **1 punto**, mientras que los participantes inactivos recibieron **0 puntos**.

Este sistema de puntuación refleja el grado de adherencia de los participantes a las recomendaciones de la OMS sobre actividad física semanal.

In [ ]:
# Assign points according to the WHO physical activity recommendations
Physical_Activity_Points = np.select(
    [
        df["Physical_Activity_Level"].isin(
            ["Meets Recommendation", "Exceeds Recommendation"]
        ),
        df["Physical_Activity_Level"] == "Below Recommendation",
        df["Physical_Activity_Level"] == "Inactive"
    ],
    [2, 1, 0],
    default=np.nan
)

## Puntuación del hábito tabáquico

El hábito tabáquico se puntuó según el estado actual de tabaquismo de los participantes.

Los participantes que nunca habían fumado recibieron la puntuación más alta (**2 puntos**). Los exfumadores recibieron **1 punto**, ya que abandonar el consumo de tabaco reduce sustancialmente los riesgos para la salud con el paso del tiempo, aunque la exposición previa al tabaco puede seguir teniendo efectos a largo plazo. Los fumadores actuales recibieron **0 puntos** debido a los efectos perjudiciales para la salud ampliamente demostrados del tabaquismo activo.

In [ ]:
# Assign points according to smoking status
Smoking_Points = np.select(
    [
        df["Smoking_Status"] == "Never smoker",
        df["Smoking_Status"] == "Former smoker",
        df["Smoking_Status"] == "Current smoker"
    ],
    [2, 1, 0],
    default=np.nan
)

## Puntuación del consumo de alcohol

El consumo de alcohol se puntuó según la frecuencia de consumo durante los 12 meses previos.

Los participantes que nunca habían consumido alcohol o que declararon un consumo raro u ocasional recibieron la puntuación más alta (**2 puntos**). Los exconsumidores y los consumidores habituales recibieron una puntuación intermedia (**1 punto**). Los participantes con un consumo frecuente o diario recibieron **0 puntos**, reflejando un patrón de consumo menos favorable.

Aunque el consumo moderado de alcohol ha sido objeto de debate en investigaciones previas, las recomendaciones actuales de salud pública enfatizan cada vez más la necesidad de minimizar su consumo, ya que ningún nivel de consumo de alcohol puede considerarse completamente exento de riesgo.

In [ ]:
# Assign points according to alcohol consumption frequency
Alcohol_Points = np.select(
    [
        df["Alcohol_Consumption"].isin(
            ["Never", "Rare", "Occasional"]
        ),
        df["Alcohol_Consumption"].isin(
            ["Former", "Regular"]
        ),
        df["Alcohol_Consumption"].isin(
            ["Frequent", "Daily"]
        )
    ],
    [2, 1, 0],
    default=np.nan
)

In [ ]:
# ==========================
# Lifestyle Profile
# ==========================

# Combine all lifestyle scores
lifestyle_points = pd.DataFrame({
    "Smoking": Smoking_Points,
    "Alcohol": Alcohol_Points,
    "Physical_Activity": Physical_Activity_Points
})

# Calculate obtained and maximum possible scores
obtained = lifestyle_points.sum(axis=1, skipna=True)
available = lifestyle_points.notna().sum(axis=1) * 2

# Percentage score
df["Lifestyle_Profile_Percentage"] = (
    obtained / available * 100
).round(2)

# Require at least 2 of the 3 indicators
df.loc[
    available < 4,
    "Lifestyle_Profile_Percentage"
] = np.nan

# Lifestyle Profile classification
df["Lifestyle_Profile"] = np.select(
    [
        df["Lifestyle_Profile_Percentage"] >= 80,
        df["Lifestyle_Profile_Percentage"].between(50, 79.9999),
        df["Lifestyle_Profile_Percentage"] < 50
    ],
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ],
    default=None
)



In [ ]:
# Classify participants according to their Lifestyle Profile
df["Lifestyle_Profile"] = np.select(
    [
        df["Lifestyle_Profile_Percentage"] >= 80,
        df["Lifestyle_Profile_Percentage"].between(50, 79.9999),
        df["Lifestyle_Profile_Percentage"] < 50
    ],
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ],
    default=None
)

display(df["Lifestyle_Profile"].value_counts(dropna=False))

### Interpretación

La clasificación del `Lifestyle_Profile` mostró que **1.893 participantes (46,14%)** fueron clasificados con un **Intermediate Lifestyle**, constituyendo el grupo de estilo de vida más frecuente de la población estudiada. Un total de **1.249 participantes (30,44%)** fueron clasificados con un **Healthy Lifestyle**, mientras que **961 participantes (23,42%)** fueron clasificados con un **Unhealthy Lifestyle**.

Los **4.103 participantes (100%)** pudieron ser clasificados en una de las tres categorías de estilo de vida. Esto fue posible gracias a que el **Lifestyle Profile** se calculó como el porcentaje de la puntuación máxima posible según los indicadores de estilo de vida disponibles para cada participante, exigiendo únicamente la disponibilidad de **al menos dos de los tres indicadores**. Dado que todos los participantes cumplían este criterio, no fue necesario excluir ninguna observación de la clasificación.

En conjunto, el `Lifestyle_Profile` proporciona un resumen integral de los principales hábitos de vida modificables de los participantes y se utilizará en los análisis posteriores para investigar si el estilo de vida está asociado con la duración del sueño.

# Análisis orientado a la hipótesis

El análisis exploratorio permitió conocer la composición de la población de estudio y la distribución de las principales variables. Sin embargo, el objetivo principal de este proyecto es evaluar si la duración del sueño está asociada con las características demográficas, los indicadores de salud y los factores relacionados con el estilo de vida.

Para garantizar un análisis estructurado, reproducible y fácil de interpretar, esta sección seguirá una matriz de análisis previamente definida. En lugar de explorar todas las combinaciones posibles de variables, cada análisis responderá a una pregunta de investigación concreta directamente relacionada con la hipótesis del estudio.

Cada relación se analizará siguiendo el mismo procedimiento:

1. Definir la pregunta de investigación.
2. Identificar el tipo de variables involucradas.
3. Seleccionar la visualización más adecuada.
4. Elegir la prueba estadística correspondiente.
5. Interpretar conjuntamente los resultados gráficos y estadísticos.
6. Resumir las conclusiones en relación con la hipótesis planteada.

Este enfoque proporciona un flujo de trabajo coherente, facilita la interpretación de los resultados y acerca el proyecto a la metodología utilizada en estudios científicos.

In [ ]:
# Pregunta de investigación 1:
# ¿La duración media del sueño difiere entre hombres y mujeres?

# Research Question 1:
# Does average sleep duration differ between males and females?

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,8))

sns.boxplot(
    data=df,
    x="Gender",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Hours by Gender")
plt.xlabel("Gender")
plt.ylabel("Average Sleep Hours")

plt.show()

print(df.groupby("Gender")["Average_Sleep_Hours"].describe())

### Interpretación

El boxplot muestra que la distribución de la duración media del sueño es similar entre mujeres y hombres. En promedio, las mujeres presentan una duración media del sueño ligeramente superior a la de los hombres (aproximadamente 18 minutos más).

La dispersión de los datos es similar en ambos grupos, aunque el rango intercuartílico de las mujeres es ligeramente mayor, lo que indica una variabilidad algo superior en el 50 % central de las observaciones. Además, se observan varios valores atípicos en ambos grupos, lo que refleja la presencia de participantes con duraciones de sueño inusualmente bajas o altas.

En conjunto, el gráfico sugiere diferencias descriptivas pequeñas entre ambos sexos, con una distribución del sueño ampliamente similar.

In [ ]:
# Research Question 2:
# Does average sleep duration differ across age groups?

# Pregunta de investigación 2:
# ¿La duración media del sueño difiere entre los distintos grupos de edad?


# Create age groups

age_bins = [20, 40, 60, float("inf")]
age_labels = [
    "Young Adults (20–39)",
    "Middle-aged Adults (40–59)",
    "Older Adults (60+)"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels,
    right=False
)

print(df.groupby('Age_Group',observed=False)['Average_Sleep_Hours'].describe())


plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="Age_Group",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Average Sleep Duration (hours)")

plt.show()

### Interpretación

Se observa que las tres categorías presentan distribuciones de la duración del sueño relativamente similares. No obstante, los adultos de mediana edad (40–59 años) muestran una duración media y una mediana del sueño ligeramente inferiores a las de los adultos jóvenes y los adultos mayores.

Por otra parte, los adultos de 60 años o más presentan una mayor variabilidad en la duración del sueño, reflejada en un rango intercuartílico más amplio y una mayor dispersión de los datos. En conjunto, las diferencias observadas entre los grupos son moderadas, aunque los adultos de mediana edad tienden a dormir ligeramente menos que el resto de los participantes.

In [ ]:
#Research Question 3:
# Does average sleep duration differ across Body Mass Index (BMI) categories?

#Pregunta de investigación 3:
# ¿La duración media del sueño difiere entre las categorías de índice de masa corporal (IMC)?

plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="BMI_Category",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across BMI Categories")
#plt.xlabel("BMI Category")
plt.xlabel("")
plt.ylabel("Average Sleep Duration (hours)")


plt.show()
print(df.groupby("BMI_Category",observed=False)["Average_Sleep_Hours"].describe())

### Interpretacion
Las cuatro categorías de IMC presentan distribuciones relativamente similares en la duración media del sueño. Sin embargo, se observa una ligera disminución de la duración media del sueño a medida que aumenta la categoría de IMC, siendo las personas con bajo peso las que presentan la media más alta y las personas con obesidad la más baja.

El grupo con bajo peso muestra la mayor variabilidad en la duración del sueño, reflejada tanto en un mayor rango intercuartílico como en una desviación estándar superior al resto de las categorías. No obstante, este grupo también presenta el menor tamaño muestral (n = 59), por lo que esta mayor dispersión debe interpretarse con cautela.

En conjunto, las diferencias entre categorías parecen moderadas y no se observan cambios muy pronunciados en la distribución de la duración del sueño.

In [ ]:
# Research Question 4:
# Does average sleep duration differ across Health Profile categories?

# Pregunta de investigación 4:
# ¿Difiere la duración promedio del sueño entre las categorías del Perfil de Salud?



plt.figure(figsize=(8,6))

sns.boxplot(
    data=df,
    x="Health_Profile",
    y="Average_Sleep_Hours"
)

plt.title("Average Sleep Duration Across Health Profile Categories")
plt.xlabel("Health Profile")
plt.ylabel("Average Sleep Duration (hours)")

plt.show()

print(df.groupby('Health_Profile')['Average_Sleep_Hours'].describe())

### Interpretacion 
Las tres categorías de perfil de salud presentan distribuciones relativamente similares en la duración del sueño. Sin embargo, se observa una ligera disminución de la duración media del sueño a medida que empeora el perfil de salud, pasando de 7.88 horas en el grupo con buen perfil de salud a 7.63 horas en el grupo con peor perfil.

El grupo con buen perfil de salud presenta una distribución ligeramente más homogénea de la duración del sueño, reflejada tanto en un rango intercuartílico algo menor como en la desviación estándar más baja (SD = 1.39). Por el contrario, el grupo con peor perfil de salud muestra una mayor dispersión de los datos (SD = 1.59), aunque las diferencias en la variabilidad entre los grupos son moderadas.

En conjunto, los resultados sugieren una tendencia hacia una menor duración del sueño y una mayor variabilidad conforme empeora el perfil de salud. 


In [ ]:
# Research Question 5:
# Does average sleep duration differ across Lifestyle Profile categories?

# Pregunta de investigación 5:
# ¿Difiere la duración promedio del sueño entre las categorías del Perfil de Estilo de Vida?


plt.figure(figsize=(8,6))
sns.boxplot(df,x='Lifestyle_Profile',y='Average_Sleep_Hours')

plt.xlabel("Lifestyle Profile")
plt.ylabel("Average Sleep Duration (hours)")
plt.title("Average Sleep Duration Across Lifestyle Profile Categories")
plt.show()

print(df.groupby("Lifestyle_Profile",observed=False)["Average_Sleep_Hours"].describe())

### Interpretación

Las distintas categorías del perfil de estilo de vida presentan, al igual que en los análisis anteriores, ligeras diferencias en la duración media del sueño que pueden interpretarse como una tendencia descriptiva. En este caso, se observa que, a medida que el perfil de estilo de vida empeora, la duración media del sueño disminuye ligeramente.

Los participantes con un **Healthy Lifestyle** presentan una duración media del sueño de **7.82 horas**, mientras que aquellos con un **Unhealthy Lifestyle** registran una media de **7.65 horas**, lo que supone una diferencia aproximada de **0.17 horas** (alrededor de **10 minutos**).

En cuanto a la variabilidad, el grupo con un estilo de vida saludable presenta la menor dispersión en la duración del sueño (SD = **1.30**), mientras que el grupo con un estilo de vida no saludable muestra la mayor variabilidad (SD = **1.73**), lo que indica una mayor heterogeneidad en los patrones de sueño de este último grupo.

En conjunto, aunque se aprecia una tendencia hacia una menor duración del sueño y una mayor variabilidad conforme empeora el estilo de vida, las diferencias observadas son relativamente pequeñas desde un punto de vista descriptivo. Será necesario realizar análisis inferenciales para determinar si estas diferencias son estadísticamente significativas.

In [ ]:
# Contingency table
health_sleep = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

display(health_sleep)

In [ ]:
health_sleep_pct = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"],
    normalize="index"
) * 100

display(health_sleep_pct.round(1))



In [ ]:
health_sleep_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(8,6)
)

plt.ylabel("Percentage")
plt.title("Health Profile Distribution Across Sleep Duration Categories")
plt.legend(title="Health Profile")
plt.xticks(rotation=0)
plt.xlabel('')
plt.show()

### Relación entre la duración del sueño y el perfil de salud

La distribución de los perfiles de salud según la duración del sueño muestra una ligera mejora a medida que aumenta la duración del sueño.

Los participantes con **Short Sleep** presentaron la menor proporción de individuos con un **Good Health Profile** (16,2%) y la mayor proporción con un **Poor Health Profile** (33,8%). En comparación, el porcentaje de participantes con un buen perfil de salud aumentó hasta el **20,0%** en la categoría **Recommended Sleep** y hasta el **21,7%** en **Long Sleep**. Paralelamente, la proporción de participantes con un perfil de salud desfavorable disminuyó del **33,8%** al **29,9%** y **29,2%**, respectivamente.

Por otro lado, la proporción de participantes con un **Intermediate Health Profile** permaneció prácticamente constante en torno al **50%** en las tres categorías de duración del sueño.

En conjunto, estos resultados sugieren una **tendencia** hacia un perfil de salud más favorable conforme aumenta la duración del sueño. Sin embargo, el **Intermediate Health Profile** siguió siendo el más frecuente en todas las categorías y, además, la proporción de participantes con un **Poor Health Profile** continuó siendo superior a la de aquellos con un **Good Health Profile**, independientemente de la duración del sueño.

# PRUEBAS CON EL MOTOR PARA ENCONTRAR INSIGHTS

In [ ]:
#LIBRERIAS
import pandas as pd
import numpy as np

from scipy.stats import (
    chi2_contingency,
    kruskal,
    spearmanr,
    pearsonr
)


In [ ]:
#BLOQUE 2 - VARIABLES
# Target variable
target = "Sleep_Duration_Category"


categorical_vars = [

    "Gender",
    "Race",
    "Education",
    "Marital_Status",
    "Age_Group",

    "Smoking_Status",
    "Alcohol_Consumption",

    "BMI_Category",

    "Physical_Activity_Level",

    "Reported_Sleep_Trouble_To_Doctor",
    "Snoring_Frequency",
    "Breathing_Interruptions_Frequency",
    "Daytime_Sleepiness_Frequency",

    "Health_Profile",
    "Lifestyle_Profile"

]


numeric_vars = [

    "Age",

    "Average_Sleep_Hours",

    "Sleep_Hours_Workdays",

    "Sleep_Hours_Weekend",

    "BMI",

    "Waist_Circumference",

    "Sedentary_Minutes_Per_Day",

    "Physical_Activity_Equivalent_Minutes_Week",

    "HbA1c",

    "HDL",

    "Systolic_BP",

    "Diastolic_BP",

    "Poverty_Index",

    "Health_Profile_Percentage",

    "Lifestyle_Profile_Percentage"

]

In [ ]:
#BLOQUE 3 - CRAMER'S V
def cramers_v(contingency):

    chi2 = chi2_contingency(contingency)[0]

    n = contingency.sum().sum()

    r, c = contingency.shape

    return np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

In [ ]:
#BLOQUE 4 - CATEGÓRICA vs CATEGÓRICA
categorical_results = []

for var in categorical_vars:

    temp = df[[target, var]].dropna()

    contingency = pd.crosstab(
        temp[target],
        temp[var]
    )

    chi2, p, dof, expected = chi2_contingency(contingency)

    cv = cramers_v(contingency)

    categorical_results.append({

        "Variable": var,

        "Test": "Chi-square",

        "Chi2": round(chi2,2),

        "p_value": round(p,5),

        "Cramers_V": round(cv,3),

        "N": len(temp)

    })

categorical_results = pd.DataFrame(categorical_results)

categorical_results.sort_values(
    "p_value",
    inplace=True
)

display(categorical_results)

In [ ]:
#BLOQUE 5 - NUMÉRICA vs SLEEP
numeric_results = []

for var in numeric_vars:

    temp = df[[target,var]].dropna()

    groups = [

        group[var].values

        for _, group in temp.groupby(target,observed=False)

    ]

    stat, p = kruskal(*groups)

    means = temp.groupby(target,observed=False)[var].mean()

    numeric_results.append({

        "Variable": var,

        "Test":"Kruskal",

        "Statistic": round(stat,2),

        "p_value": round(p,5),

        "Short_Mean": round(means.get("Short Sleep",np.nan),2),

        "Recommended_Mean": round(means.get("Recommended Sleep",np.nan),2),

        "Long_Mean": round(means.get("Long Sleep",np.nan),2),

        "N": len(temp)

    })

numeric_results = pd.DataFrame(numeric_results)

numeric_results.sort_values(
    "p_value",
    inplace=True
)

display(numeric_results)

In [ ]:
#BLOQUE 6 CORRELACIONES
correlations = []

for i in range(len(numeric_vars)):

    for j in range(i+1,len(numeric_vars)):

        var1 = numeric_vars[i]

        var2 = numeric_vars[j]

        temp = df[[var1,var2]].dropna()

        corr,p = spearmanr(
            temp[var1],
            temp[var2]
        )

        correlations.append({

            "Variable_1":var1,

            "Variable_2":var2,

            "Correlation":round(corr,3),

            "p_value":round(p,5)

        })

correlations = pd.DataFrame(correlations)

correlations["abs_corr"] = correlations["Correlation"].abs()

correlations = correlations.sort_values(
    "abs_corr",
    ascending=False
)

display(correlations.head(20))

In [ ]:
#TODO CON TODO 

all_categorical = categorical_vars + [target]

association_matrix = []

for i in range(len(all_categorical)):

    for j in range(i+1,len(all_categorical)):

        var1 = all_categorical[i]

        var2 = all_categorical[j]

        temp = df[[var1,var2]].dropna()

        contingency = pd.crosstab(
            temp[var1],
            temp[var2]
        )

        chi2,p,dof,expected = chi2_contingency(contingency)

        cv = cramers_v(contingency)

        association_matrix.append({

            "Variable_1":var1,

            "Variable_2":var2,

            "p_value":round(p,5),

            "Cramers_V":round(cv,3)

        })

association_matrix = pd.DataFrame(association_matrix)

association_matrix = association_matrix.sort_values(

    ["Cramers_V","p_value"],

    ascending=[False,True]

)

display(association_matrix.head(50))

# Exploracion de Variabilidad y Proporcion de categorias de sueño 

In [ ]:
#Districucion del IMC en las categorias de sueño
import seaborn as sns

plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="Sleep_Duration_Category",
    y="BMI",
    order=["Short Sleep","Recommended Sleep","Long Sleep"]
)

plt.xlabel("Sleep Duration Category")
plt.ylabel("BMI")
plt.tight_layout()

plt.show()

In [ ]:
#Porcentaje o concentracion de categorias de sueño segun perfiles de salud

import matplotlib.pyplot as plt

health_percent = (
    pd.crosstab(
        df["Health_Profile"],
        df["Sleep_Duration_Category"],
        normalize="index"
    ) * 100
).round(1)



ax = health_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(12,5)
)

plt.ylabel("Percentage (%)")
plt.xlabel("Sleep Duration Category")
plt.xticks(rotation=0)
plt.legend(title="Health Profile", bbox_to_anchor=(1.02,1), loc="upper left")

# Añadir etiquetas
for container in ax.containers:
    labels = [
        f"{bar.get_height():.1f}%"
        if bar.get_height() > 3 else ""
        for bar in container
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=9)

plt.tight_layout()
plt.show()

## Pregunta de investigación 1

### ¿Se asocia una menor duración del sueño con un peor perfil de salud?

Para responder esta pregunta se analizó la relación entre la categoría de duración del sueño y el perfil de salud de los participantes. En primer lugar, se examinó la distribución porcentual de los perfiles de salud dentro de cada categoría de duración del sueño. Posteriormente, se evaluó estadísticamente la asociación entre ambas variables mediante la prueba Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# ============================================================
# Research Question 1
# Is shorter sleep duration associated with a poorer health profile?
# ============================================================

import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np

# ------------------------------------------------------------
# Percentage table
# ------------------------------------------------------------

health_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Health_Profile"],
        normalize="index"
    ) * 100
).round(1)

display(health_percent)

# ------------------------------------------------------------
# Percentage plot
# ------------------------------------------------------------

ax = health_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(9,6)
)

plt.title("Health Profile Distribution by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)

plt.legend(
    title="Health Profile",
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)

# Add percentage labels

for container in ax.containers:

    labels = [

        f"{bar.get_height():.1f}%"
        if bar.get_height() > 3
        else ""

        for bar in container

    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9
    )

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Chi-square test
# ------------------------------------------------------------

contingency = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency)

# ------------------------------------------------------------
# Cramer's V
# ------------------------------------------------------------

n = contingency.sum().sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print("=" * 40)
print("Statistical Results")
print("=" * 40)

print(f"Chi-square : {chi2:.2f}")
print(f"Degrees of freedom : {dof}")
print(f"P-value : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

In [ ]:
#Presentacion
# ============================================================
# Health Profile by Sleep Duration (Grouped Bar Chart)
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np

# ------------------------------------------------------------
# Percentage table
# ------------------------------------------------------------

health_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Health_Profile"],
        normalize="index"
    ) * 100
).round(1)

print(health_percent)

# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------

colors = {
    "Good Health Profile": "#2ca02c",          # Verde
    "Intermediate Health Profile": "#ff7f0e",  # Naranja
    "Poor Health Profile": "#d62728"           # Rojo
}
# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10,6))

health_percent.plot(
    kind="bar",
    ax=ax,
    color=colors,
    width=0.8
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

ax.set_title(
    "Health Profile by Sleep Duration",
    fontsize=15,
    weight="bold"
)

ax.set_xlabel("Sleep Duration Category")
ax.set_ylabel("Percentage (%)")

ax.set_ylim(0,60)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.3
)

plt.xticks(rotation=0)

# ------------------------------------------------------------
# Labels on bars
# ------------------------------------------------------------

for container in ax.containers:

    labels = [
        f"{bar.get_height():.1f}%"
        if bar.get_height() >= 3
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        padding=3,
        fontsize=9
    )

# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

ax.legend(
    [
        "Good Health",
        "Intermediate Health",
        "Poor Health"
    ],
    title="Health Profile",
    frameon=False,
    bbox_to_anchor=(1.02,1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Chi-square
# ------------------------------------------------------------

contingency = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Health_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency)

# ------------------------------------------------------------
# Cramer's V
# ------------------------------------------------------------

n = contingency.values.sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r-1, c-1))
)

print("="*40)
print("Statistical Results")
print("="*40)
print(f"Chi-square : {chi2:.2f}")
print(f"P-value    : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

### Interpretación

El gráfico muestra que las diferencias entre las categorías de duración del sueño no son muy pronunciadas; sin embargo, puede apreciarse un patrón consistente. A medida que el perfil de salud pasa de **Good Health Profile** a **Poor Health Profile**, aumenta ligeramente la proporción de participantes con **Short Sleep** y disminuye la proporción de aquellos con una duración de sueño recomendada.

Este patrón es coherente con los resultados de la prueba Chi-cuadrado, que mostró una asociación estadísticamente significativa entre la duración del sueño y el perfil de salud (χ² = 11.05, *p* = 0.026). No obstante, el valor de **V de Cramér = 0.038** indica que la fuerza de esta asociación es **muy débil**, por lo que, aunque existe una tendencia, la duración del sueño por sí sola explica una pequeña parte de las diferencias observadas en el perfil de salud de los participantes.

## Pregunta de investigación 2

### ¿Se asocia una menor duración del sueño con un peor perfil de estilo de vida?

Para responder esta pregunta se analizó la relación entre la categoría de duración del sueño y el perfil de estilo de vida de los participantes. En primer lugar, se examinó la distribución porcentual de los perfiles de estilo de vida dentro de cada categoría de duración del sueño. Posteriormente, se evaluó estadísticamente la asociación entre ambas variables mediante la prueba Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of Lifestyle Profile by Sleep Duration Category
lifestyle_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Lifestyle_Profile"],
        normalize="index"
    ) * 100
).round(1)

# Order categories from healthiest to least healthy
lifestyle_percent = lifestyle_percent[
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ]
]

print(lifestyle_percent)

# Consistent project color palette


colors = [
     "#2ca02c",   # Verde Healthy Lifestyle
     "#ff7f0e",         # Naranja  recomend
     "#1f77b4"           # Azul   
]

# Create figure
fig, ax = plt.subplots(figsize=(9, 5.5))

lifestyle_percent.plot(
    kind="bar",
    stacked=True,
    color=colors,
    edgecolor="white",
    width=0.7,
    ax=ax
)

# Add percentage labels (only >= 5%)
for container in ax.containers:
    labels = [
        f"{value:.1f}%"
        if value >= 5 else ""
        for value in container.datavalues
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9,
        color="black"
    )

# Formatting
ax.set_title(
    "Lifestyle Profile Distribution by Sleep Duration Category",
    fontsize=14,
    weight="bold",
    pad=15
)

ax.set_xlabel("Sleep Duration Category", fontsize=11)
ax.set_ylabel("Participants (%)", fontsize=11)

# Shorter x-axis labels
ax.set_xticklabels([
    "Short",
    "Recommended",
    "Long"
], rotation=0, fontsize=10)

ax.tick_params(axis="y", labelsize=10)

# Grid
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.grid(axis="x", visible=False)

# Legend
ax.legend(
    title="Lifestyle Profile",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=10,
    title_fontsize=11
)

plt.tight_layout()
plt.show()

# ----------------------------
# Chi-square Test
# ----------------------------
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Lifestyle_Profile"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Percentage distribution
lifestyle_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Lifestyle_Profile"],
        normalize="index"
    ) * 100
).round(1)

# Order rows
lifestyle_percent = lifestyle_percent[
    [
        "Healthy Lifestyle",
        "Intermediate Lifestyle",
        "Unhealthy Lifestyle"
    ]
]

# Normalize values (0–1) for the color scale
heatmap_data = lifestyle_percent / 100

# Plot
plt.figure(figsize=(7, 4.8))

ax = sns.heatmap(
    heatmap_data,
    annot=lifestyle_percent.astype(str) + "%",
    fmt="",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Proportion"}
)

# Labels
ax.set_title(
    "Lifestyle Profile by Sleep Duration Category",
    fontsize=14,
    weight="bold"
)

ax.set_xlabel("Lifestyle Profile")
ax.set_ylabel("Sleep Duration Category")

ax.set_xticklabels(
    ["Healthy", "Intermediate", "Unhealthy"],
    rotation=0
)

ax.set_yticklabels(
    ["Short", "Recommended", "Long"],
    rotation=0
)

plt.tight_layout()
plt.show()

### Interpretación

La distribución porcentual sugiere una relación consistente, aunque sutil, entre la duración del sueño y el perfil de estilo de vida. Los participantes con una **duración de sueño recomendada** presentaron la mayor proporción de **Healthy Lifestyle** (33.1%) y la menor proporción de **Unhealthy Lifestyle** (21.6%). En contraste, los participantes con **Short Sleep** mostraron la menor proporción de estilos de vida saludables (26.9%) y la mayor proporción de estilos de vida no saludables (26.0%). Por su parte, los participantes con **Long Sleep** presentaron un patrón intermedio, con porcentajes más próximos a los observados en el grupo de sueño corto.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y el perfil de estilo de vida (χ²(4) = 20.92, *p* < 0.001). Sin embargo, el **V de Cramér = 0.051** indica que la fuerza de esta asociación es **muy débil**. Por tanto, aunque existe una relación entre ambas variables, la duración del sueño explica únicamente una pequeña parte de las diferencias observadas en el perfil de estilo de vida de los participantes.

Para comprender mejor esta asociación, en los siguientes apartados se analizarán individualmente los principales componentes del perfil de estilo de vida, incluyendo el tabaquismo, el nivel de actividad física y el consumo de alcohol.

### Nivel de actividad física

Para profundizar en la relación entre la duración del sueño y el estilo de vida, se analizó la asociación entre la categoría de duración del sueño y el nivel de actividad física de los participantes. En primer lugar, se exploró la distribución porcentual de los niveles de actividad física dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of physical activity level by sleep duration category
pa_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Physical_Activity_Level"],
        normalize="index"
    ) * 100
).round(1)

display(pa_percent)

colors = [
    "#d73027",  # Inactive
    "#fc8d59",  # Below Recommendation
    "#91cf60",  # Meets Recommendation
    "#1a9850"   # Exceeds Recommendation
]

# Create stacked percentage bar chart
ax = pa_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5),
    color=colors
)


# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

# Customize chart
plt.title("Physical Activity Level by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)
plt.legend(
    title="Physical Activity Level",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Physical_Activity_Level"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra una relación sutil entre la duración del sueño y el nivel de actividad física. Los participantes con una **duración de sueño recomendada** presentaron la menor proporción de personas inactivas (40.9%) y la mayor proporción de participantes que superaban las recomendaciones de actividad física (30.2%). En contraste, los participantes con **Long Sleep** mostraron la mayor proporción de personas inactivas (48.4%) y la menor proporción de participantes que superaban las recomendaciones (23.2%). El grupo con **Short Sleep** presentó un patrón intermedio.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y el nivel de actividad física (χ²(6) = 19.11, *p* = 0.004). Sin embargo, el **V de Cramér = 0.049** indica que la fuerza de esta asociación es **muy débil**. Por tanto, aunque el nivel de actividad física varía entre las distintas categorías de duración del sueño, la magnitud de estas diferencias es pequeña.

En conjunto, estos resultados sugieren que los participantes con una duración de sueño recomendada tienden a presentar un patrón de actividad física más favorable, mientras que aquellos con una duración de sueño prolongada muestran una mayor tendencia a la inactividad física.

### Estado de tabaquismo

Para continuar analizando los componentes del perfil de estilo de vida, se examinó la relación entre la categoría de duración del sueño y el estado de tabaquismo de los participantes. En primer lugar, se exploró la distribución porcentual de las diferentes categorías de tabaquismo dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of smoking status by sleep duration category
smoking_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Smoking_Status"],
        normalize="index"
    ) * 100
).round(1)

print(smoking_percent)

# Color palette (healthier → greener)
colors = [
    "#d73027",  # Current smoker (red)
    "#fc8d59",  # Former smoker (orange)
    "#1a9850"   # Never smoker (green)
]

# Create stacked percentage bar chart
ax = smoking_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=colors,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

# Customize chart
plt.title("Smoking Status by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)
plt.legend(
    title="Smoking Status",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Smoking_Status"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra una relación consistente entre la duración del sueño y el estado de tabaquismo. Los participantes con una **duración de sueño recomendada** presentaron la mayor proporción de **Never smokers** (59.0%) y la menor proporción de **Current smokers** (16.4%). En contraste, los participantes con **Short Sleep** mostraron la mayor proporción de fumadores actuales (24.0%) y la menor proporción de personas que nunca habían fumado (52.1%). Los participantes con **Long Sleep** presentaron un patrón intermedio.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y el estado de tabaquismo (χ²(4) = 28.69, *p* < 0.001). Sin embargo, el **V de Cramér = 0.059** indica que la fuerza de esta asociación es **muy débil**. Por tanto, aunque el estado de tabaquismo varía entre las distintas categorías de duración del sueño, la magnitud de estas diferencias es pequeña.

En conjunto, estos resultados sugieren que los participantes con una duración de sueño recomendada presentan un patrón de tabaquismo más favorable, mientras que aquellos con una duración de sueño corta muestran una mayor proporción de fumadores actuales.

### Consumo de alcohol

Para completar el análisis del perfil de estilo de vida, se examinó la relación entre la categoría de duración del sueño y el nivel de consumo de alcohol de los participantes. En primer lugar, se exploró la distribución porcentual de las diferentes categorías de consumo de alcohol dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of alcohol consumption by sleep duration category
alcohol_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Alcohol_Consumption"],
        normalize="index"
    ) * 100
).round(1)

display(alcohol_percent)

# Color palette
colors = [
    "#1a9850",  # Never
    "#91cf60",  # Former
    "#fee08b",  # Rare
    "#fdae61",  # Occasional
    "#fc8d59",  # Regular
    "#d73027",  # Frequent
    "#a50026"   # Daily
]

# Create stacked percentage bar chart
ax = alcohol_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=colors,
    width=0.65,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center", fontsize=8)

# Customize chart
plt.title("Alcohol Consumption by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Alcohol Consumption",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Alcohol_Consumption"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency_table.shape) - 1)))

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra diferencias modestas en los patrones de consumo de alcohol entre las distintas categorías de duración del sueño. Los participantes con **Long Sleep** presentaron una mayor proporción de **Former drinkers** (31.5%) y una menor proporción de **Regular drinkers** (10.7%) en comparación con los otros grupos de duración del sueño. En contraste, las distribuciones correspondientes a **Short Sleep** y **Recommended Sleep** fueron relativamente similares en la mayoría de las categorías de consumo de alcohol.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y el consumo de alcohol (χ²(12) = 30.90, *p* = 0.002). Sin embargo, el **V de Cramér = 0.062** indica que la fuerza de esta asociación es **débil**. Por tanto, aunque los patrones de consumo de alcohol varían entre las distintas categorías de duración del sueño, la magnitud de estas diferencias es limitada.

En conjunto, estos resultados sugieren que el consumo de alcohol está asociado con la duración del sueño. No obstante, la fuerza de esta relación es relativamente débil, por lo que debe interpretarse con cautela.

### Frecuencia de ronquidos

Para investigar la posible relación entre la duración del sueño y los síntomas relacionados con el sueño, se analizó la asociación entre la categoría de duración del sueño y la frecuencia de ronquidos. En primer lugar, se exploró la distribución porcentual de la frecuencia de ronquidos dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of snoring frequency by sleep duration category
snoring_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Snoring_Frequency"],
        normalize="index"
    ) * 100
).round(1)

# Reorder categories from lowest to highest frequency
snoring_percent = snoring_percent[
    [
        "Never",
        "Rarely (1-2 nights/week)",
        "Occasionally (3-4 nights/week)",
        "Frequently (5+ nights/week)"
    ]
]

display(snoring_percent)

# Color palette
colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Occasionally
    "#d73027"   # Frequently
]

# Create stacked percentage bar chart
ax = snoring_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    width=0.7,
    color=colors,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 5 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

# Customize chart
plt.title("Snoring Frequency by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Snoring Frequency",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Snoring_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra diferencias modestas en la frecuencia de ronquidos entre las distintas categorías de duración del sueño. Los participantes con **Recommended Sleep** presentaron la menor proporción de individuos que reportaron ronquidos frecuentes (27.9%), mientras que los grupos de **Short Sleep** (33.2%) y **Long Sleep** (33.4%) mostraron porcentajes similares y más elevados de ronquidos frecuentes. En las demás categorías de frecuencia de ronquidos se observaron diferencias relativamente pequeñas entre los tres grupos de duración del sueño.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y la frecuencia de ronquidos (χ²(6) = 16.07, *p* = 0.013). Sin embargo, el **V de Cramér = 0.046** indica que la fuerza de esta asociación es **muy débil**. Por tanto, aunque la frecuencia de ronquidos varía entre las distintas categorías de duración del sueño, la magnitud de estas diferencias es limitada.

En conjunto, estos resultados sugieren que los participantes con una duración de sueño recomendada presentan una menor proporción de ronquidos frecuentes en comparación con aquellos que duermen menos o más horas de las recomendadas. No obstante, la relación observada entre la duración del sueño y la frecuencia de ronquidos es débil.

### Frecuencia de interrupciones respiratorias durante el sueño

Para profundizar en el análisis de los problemas relacionados con el sueño, se examinó la asociación entre la categoría de duración del sueño y la frecuencia de interrupciones respiratorias durante el sueño. En primer lugar, se exploró la distribución porcentual de la frecuencia de interrupciones respiratorias dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of breathing interruptions by sleep duration category
breathing_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Breathing_Interruptions_Frequency"],
        normalize="index"
    ) * 100
).round(1)

# Reorder categories from lowest to highest frequency
breathing_percent = breathing_percent[
    [
        "Never",
        "Rarely (1-2 nights/week)",
        "Occasionally (3-4 nights/week)",
        "Frequently (5+ nights/week)"
    ]
]

print(breathing_percent)

# Color palette
colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Occasionally
    "#d73027"   # Frequently
]

# Create stacked percentage bar chart
ax = breathing_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    width=0.7,
    color=colors,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value >= 1 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

# Customize chart
plt.title("Breathing Interruptions Frequency by Sleep Duration Category", fontsize=14)
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Breathing Interruptions Frequency",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Breathing_Interruptions_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra diferencias modestas en la frecuencia de interrupciones respiratorias entre las distintas categorías de duración del sueño. Los participantes con **Recommended Sleep** presentaron la mayor proporción de individuos que reportaron **nunca** experimentar interrupciones respiratorias durante el sueño (76.8%) y la menor proporción de quienes las experimentaban **frecuentemente** (4.9%). En comparación, los grupos de **Short Sleep** y **Long Sleep** mostraron porcentajes ligeramente superiores de interrupciones respiratorias frecuentes.

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y la frecuencia de interrupciones respiratorias (χ²(6) = 17.27, *p* = 0.008). Sin embargo, el **V de Cramér = 0.048** indica que la fuerza de esta asociación es **muy débil**. Por tanto, aunque la frecuencia de las interrupciones respiratorias varía entre las distintas categorías de duración del sueño, la magnitud de estas diferencias es limitada.

En conjunto, estos resultados sugieren que los participantes con una duración de sueño recomendada presentan un patrón ligeramente más favorable en cuanto a las interrupciones respiratorias durante el sueño. No obstante, la relación observada entre ambas variables es débil y debe interpretarse con cautela.

In [ ]:
df
display(df.groupby('Marital_Status')['Average_Sleep_Hours'].mean().sort_values(ascending=False).round(2))

### Problemas de sueño reportados a un profesional de la salud

Con el objetivo de evaluar si la duración del sueño se relaciona con la percepción de problemas de sueño que requieren atención sanitaria, se examinó la asociación entre la categoría de duración del sueño y el hecho de haber reportado problemas de sueño a un profesional de la salud. En primer lugar, se exploró la distribución porcentual de esta variable dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of reported sleep trouble by sleep duration category
sleep_trouble_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Reported_Sleep_Trouble_To_Doctor"],
        normalize="index"
    ) * 100
).round(1)

# Reorder categories
sleep_trouble_percent = sleep_trouble_percent[
    [
        "No",
        "Yes"
    ]
]

print(sleep_trouble_percent)

# Color palette
colors = [
    "#1a9850",  # No
    "#d73027"   # Yes
]

# Create stacked percentage bar chart
ax = sleep_trouble_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    color=colors,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

# Customize chart
plt.title("Reported Sleep Trouble by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Reported Sleep Trouble",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Reported_Sleep_Trouble_To_Doctor"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra diferencias moderadas en la proporción de participantes que reportaron haber consultado a un profesional de la salud por problemas de sueño según la categoría de duración del sueño. Los participantes con **Short Sleep** presentaron el mayor porcentaje de personas que habían reportado problemas de sueño a un profesional sanitario (32.8%), mientras que aquellos con **Recommended Sleep** mostraron el menor porcentaje (26.9%). El grupo de **Long Sleep** presentó una proporción intermedia (29.6%).

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y el hecho de haber reportado problemas de sueño a un profesional de la salud (χ²(2) = 12.41, *p* = 0.002). Sin embargo, el **V de Cramér = 0.055** indica que la fuerza de esta asociación es **muy débil**. Esto sugiere que, aunque los participantes con una duración de sueño corta reportan con mayor frecuencia problemas de sueño, la magnitud de la asociación es limitada.

En conjunto, estos resultados respaldan la hipótesis de que una duración de sueño inferior a la recomendada se relaciona con una mayor probabilidad de reportar problemas de sueño a un profesional sanitario. No obstante, la fuerza de esta relación es reducida y debe interpretarse con cautela.

### Frecuencia de somnolencia diurna

Finalmente, se analizó la asociación entre la categoría de duración del sueño y la frecuencia con la que los participantes experimentaban somnolencia durante el día. En primer lugar, se exploró la distribución porcentual de la frecuencia de somnolencia diurna dentro de cada categoría de duración del sueño. Posteriormente, la asociación entre ambas variables se evaluó mediante la prueba de Chi-cuadrado y el coeficiente V de Cramér.

In [ ]:
# Percentage distribution of daytime sleepiness frequency by sleep duration category
sleepiness_percent = (
    pd.crosstab(
        df["Sleep_Duration_Category"],
        df["Daytime_Sleepiness_Frequency"],
        normalize="index"
    ) * 100
).round(1)

# Reorder categories
sleepiness_percent = sleepiness_percent[
    [
        "Never",
        "Rarely (1 time/month)",
        "Sometimes (2-4 times/month)",
        "Often (5-15 times/month)",
        "Almost always (16-30 times/month)"
    ]
]

print(sleepiness_percent)

# Color palette (least to most frequent)
colors = [
    "#1a9850",  # Never
    "#91cf60",  # Rarely
    "#fdae61",  # Sometimes
    "#fc8d59",  # Often
    "#d73027"   # Almost always
]

# Create stacked percentage bar chart
ax = sleepiness_percent.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5),
    color=colors,
    edgecolor="white"
)

# Add percentage labels
for container in ax.containers:
    labels = [
        f"{value:.1f}%" if value > 0 else ""
        for value in container.datavalues
    ]
    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

# Customize chart
plt.title("Daytime Sleepiness Frequency by Sleep Duration Category")
plt.xlabel("Sleep Duration Category")
plt.ylabel("Percentage of Participants")
plt.xticks(rotation=0)

plt.legend(
    title="Daytime Sleepiness Frequency",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# Chi-square test
contingency_table = pd.crosstab(
    df["Sleep_Duration_Category"],
    df["Daytime_Sleepiness_Frequency"]
)

chi2, p, dof, expected = chi2_contingency(contingency_table)

# Calculate Cramer's V
n = contingency_table.to_numpy().sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

# Print statistical results
print(f"Chi-square test: χ²({dof}) = {chi2:.2f}")
print(f"P-value: {p:.4f}")
print(f"Cramer's V: {cramers_v:.3f}")

### Interpretación

La distribución porcentual muestra diferencias en la frecuencia de somnolencia diurna entre las distintas categorías de duración del sueño. Los participantes con **Long Sleep** presentaron la mayor proporción de individuos que **nunca** experimentaban somnolencia diurna (18.0%). Sin embargo, los participantes con **Recommended Sleep** mostraron la menor proporción de quienes experimentaban somnolencia diurna **casi siempre** (6.6%), mientras que el grupo de **Short Sleep** presentó el mayor porcentaje de somnolencia diurna muy frecuente (13.4%).

La prueba de Chi-cuadrado indicó una asociación estadísticamente significativa entre la duración del sueño y la frecuencia de somnolencia diurna (χ²(8) = 52.12, *p* < 0.001). El **V de Cramér = 0.080** indica que la fuerza de esta asociación es **débil**, aunque representa la asociación más fuerte observada entre las variables relacionadas con problemas del sueño analizadas en este estudio.

En conjunto, estos resultados sugieren que la duración del sueño se asocia con la frecuencia de somnolencia diurna. En particular, los participantes con **Short Sleep** presentaron una mayor proporción de somnolencia diurna muy frecuente, mientras que aquellos con **Recommended Sleep** mostraron la menor proporción de este patrón. Aunque la magnitud de la asociación sigue siendo limitada, este hallazgo respalda la hipótesis de que la duración del sueño está relacionada con el funcionamiento durante el día.

# Research Question 4: Demographic Factors

## Which demographic factors are associated with sleep duration among U.S. adults?

This section investigates whether demographic characteristics are associated with sleep duration categories. For each demographic variable, the analysis includes a contingency table, percentage distribution, Chi-square test of independence, Cramer's V effect size, and an appropriate visualization. The objective is to identify which demographic factors show the strongest association with sleep duration among participants in the NHANES 2017–2018 dataset.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def analyze_categorical(df, variable, target="Sleep_Duration_Category"):
    """
    Perform a categorical association analysis between a predictor
    and the sleep duration category.
    """

    # Remove missing values
    data = df[[variable, target]].dropna()

    # Contingency table
    contingency = pd.crosstab(data[variable], data[target])

    # Percentage table
    percentages = (
        pd.crosstab(
            data[variable],
            data[target],
            normalize="index"
        ) * 100
    ).round(1)
    
    frequencies = (
    data[variable]
    .value_counts()
    .sort_index()
    )

    # Chi-square test
    chi2, p, dof, expected = chi2_contingency(contingency)

    # Cramer's V
    n = contingency.values.sum()
    cramers_v = np.sqrt(
        chi2 / (n * (min(contingency.shape) - 1))
    )

    return {
    "frequencies": frequencies,
    "contingency": contingency,
    "percentages": percentages,
    "chi2": chi2,
    "p": p,
    "dof": dof,
    "cramers_v": cramers_v,
    "expected": expected
    }

### Género

Se analizó la variable **Gender** para evaluar si la distribución de las categorías de duración del sueño difiere entre hombres y mujeres. Para ello, se construyó una tabla de contingencia con sus porcentajes por fila y se aplicó la prueba de independencia Chi-cuadrado junto con el estadístico V de Cramer's para medir la fuerza de la asociación. Además, se utilizó un violin plot para comparar la distribución de las horas promedio de sueño entre ambos géneros.

In [ ]:
gender = analyze_categorical(df, "Gender")
print(gender["contingency"])

In [ ]:
print(gender["percentages"])

In [ ]:
print(f"Chi-square : {gender['chi2']:.2f}")
print(f"Degrees of freedom : {gender['dof']}")
print(f"p-value : {gender['p']:.4f}")
print(f"Cramer's V : {gender['cramers_v']:.3f}")

In [ ]:
print(df.groupby("Gender")["Average_Sleep_Hours"].describe())

In [ ]:
# Presentación
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

ax = sns.violinplot(
    data=df,
    x="Gender",
    y="Average_Sleep_Hours",
    palette=["#F53809", "#241DAD"],
    inner="box",      # Muestra el boxplot (mediana + cuartiles)
    cut=0,
    linewidth=1.2
)

plt.title(
    "Distribución de la duración del sueño por sexo",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("")
plt.ylabel("Horas medias de sueño")

plt.grid(axis="y", linestyle="--", alpha=0.3)

sns.despine()

plt.tight_layout()
plt.show()

### Interpretacion

El análisis mostró una asociación estadísticamente significativa entre el género y las categorías de duración del sueño (χ² = 36.49, *p* < 0.001). Los hombres presentaron una mayor proporción de **Short Sleep** (28.5%) en comparación con las mujeres (21.5%), mientras que las mujeres mostraron porcentajes ligeramente superiores tanto en **Recommended Sleep** (61.9% frente a 59.6%) como en **Long Sleep** (16.6% frente a 11.9%).

Sin embargo, el tamaño del efecto fue **muy débil** (V de Cramer's = 0.095), lo que indica que, aunque existe una asociación estadísticamente significativa entre el género y la duración del sueño, esta explica solo una pequeña parte de las diferencias observadas.

El violin plot complementa estos resultados al mostrar que la distribución de las horas promedio de sueño es muy similar entre ambos géneros. No obstante, la distribución femenina se encuentra ligeramente desplazada hacia valores superiores, con una media (7.90 h) y una mediana (8.00 h) mayores que las observadas en los hombres (7.60 h y 7.57 h, respectivamente). Además, la variabilidad fue prácticamente idéntica en ambos grupos (DE ≈ 1.51 h), lo que sugiere patrones de sueño similares, aunque las mujeres tienden a dormir ligeramente más en promedio.

### Grupo de edad

Se analizó la variable **Age_Group_10** para evaluar si la distribución de las categorías de duración del sueño difiere entre los distintos grupos de edad. Para ello, se construyó una tabla de contingencia con sus porcentajes por fila y se aplicó la prueba de independencia Chi-cuadrado junto con el estadístico V de Cramer's para medir la fuerza de la asociación.

In [ ]:
bins = [20, 30, 40, 50, 60, 70, 81]
labels = ["20–29", "30–39", "40–49", "50–59", "60–69", "70–80"]

df["Age_Group_10"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels,
    right=False
)

analyze_categorical(df, "Age_Group_10")

In [ ]:
print(df["Age_Group_10"].value_counts().sort_index())
age = analyze_categorical(df, "Age_Group_10")
print(age["percentages"])

In [ ]:
#Para presentacion
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))


colors = {
    "Recommended Sleep": "#2ca02c",   # Verde
    "Short Sleep": "#ff7f0e",         # Naranja
    "Long Sleep": "#1f77b4"           # Azul
}

age["percentages"].plot(
    kind="bar",
    ax=ax,
    width=0.8,
    color=[colors[col] for col in age["percentages"].columns]
)

# Add percentage labels on each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=9, padding=2)

# Title and labels
ax.set_title("Distribución de la duración del sueño según el grupo de edad", fontsize=14)
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("Porcentaje (%)")
ax.set_ylim(0, 70)

# Grid
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Custom legend
ax.legend(
    ["Sueño corto (<7 h)",
        "Sueño recomendado (7–9 h)",

        "Sueño largo (>9 h)"
    ],
    title="Categorías de sueño",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False
)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import chi2_contingency
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Contingency table
# ------------------------------------------------------------

contingency = pd.crosstab(
    df["Age_Group"],
    df["Sleep_Duration_Category"]
)

# ------------------------------------------------------------
# Chi-square test
# ------------------------------------------------------------

chi2, p, dof, expected = chi2_contingency(contingency)

# ------------------------------------------------------------
# Cramer's V
# ------------------------------------------------------------

n = contingency.values.sum()

r, c = contingency.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print("="*40)
print("Statistical Results")
print("="*40)
print(f"Chi-square : {chi2:.2f}")
print(f"P-value    : {p:.4f}")
print(f"Cramer's V : {cramers_v:.3f}")

### Interpretación

El gráfico muestra que las diferencias entre las categorías de duración del sueño no son muy pronunciadas; sin embargo, puede apreciarse un patrón consistente. A medida que el perfil de salud pasa de **Good Health Profile** a **Poor Health Profile**, aumenta ligeramente la proporción de participantes con **Short Sleep** y disminuye la proporción de aquellos con una duración de sueño recomendada.

Este patrón es coherente con los resultados de la prueba Chi-cuadrado, que mostró una asociación estadísticamente significativa entre la duración del sueño y el perfil de salud (χ² = 11.05, *p* = 0.026). No obstante, el valor de **V de Cramér = 0.038** indica que la fuerza de esta asociación es **muy débil**, por lo que, aunque existe una tendencia, la duración del sueño por sí sola explica una pequeña parte de las diferencias observadas en el perfil de salud de los participantes.

Para comprender mejor esta relación, en los siguientes apartados se analizarán de forma individual indicadores de salud como el IMC, la circunferencia de cintura y la HbA1c.

### Distribución de la Duración del Sueño según el Grupo de Edad

La distribución de las categorías de duración del sueño muestra patrones diferentes según el grupo de edad. En primer lugar, la proporción de participantes con una **duración de sueño recomendada** se mantiene relativamente estable en todos los grupos etarios, situándose en torno al 60%. No obstante, se observa un descenso más pronunciado en el grupo de **60–69 años** (57.6%), seguido de una recuperación en el grupo de **70–80 años** (62.3%).

En cuanto a la categoría de **Short Sleep**, se aprecia una tendencia ascendente desde el grupo de **20–29 años** (20.1%) hasta alcanzar su valor máximo en el grupo de **50–59 años** (29.4%). A partir de esa edad, la proporción disminuye progresivamente hasta situarse en **18.4%** en el grupo de **70–80 años**.

Por el contrario, la categoría de **Long Sleep** presenta un patrón inverso. Su porcentaje disminuye desde **17.0%** en el grupo de **20–29 años** hasta un mínimo de **9.4%** en el grupo de **50–59 años**, para posteriormente aumentar de nuevo hasta **19.2%** en el grupo de **70–80 años**.

En conjunto, estos resultados sugieren que las categorías de **Short Sleep** y **Long Sleep** siguen tendencias opuestas a lo largo de los distintos grupos de edad, mientras que la proporción de participantes con una **duración de sueño recomendada** permanece relativamente constante durante la edad adulta.

### Raza / Etnia

Este análisis evalúa si la duración del sueño difiere entre los distintos grupos raciales y étnicos de la muestra. En primer lugar, se presenta la distribución de participantes por raza/etnia, seguida de un mapa de calor que muestra el porcentaje de cada categoría de duración del sueño dentro de cada grupo. Finalmente, se aplica la prueba Chi-cuadrado de independencia y el estadístico V de Cramér para evaluar la significancia estadística y la fuerza de la asociación entre la raza/etnia y la duración del sueño.

In [ ]:
race_results = analyze_categorical(df_analysis, "Race")

In [ ]:
# Frequency table
display(race_results["contingency"].sum(axis=1))

In [ ]:
# Contingency table
display(race_results["contingency"])

In [ ]:
# Row percentages
display(race_results["percentages"])

In [ ]:
import seaborn as sns

plt.figure(figsize=(8,4))

sns.heatmap(
    race_results["percentages"],
    annot=True,
    cmap="Blues",
    fmt=".1f"
)

plt.title("Sleep Duration Category by Race (%)")
plt.xlabel("")
plt.ylabel("")
plt.show()

In [ ]:
print(f"Chi²: {race_results['chi2']:.2f}")
print(f"p-value: {race_results['p']:.4f}")
print(f"Cramer's V: {race_results['cramers_v']:.3f}")

### Interpretacion

El mapa de calor muestra diferencias apreciables en la distribución de la duración del sueño entre los distintos grupos raciales y étnicos. Los participantes **Non-Hispanic Asian** presentaron la mayor proporción de **Recommended Sleep** (68.9%) y la menor proporción de **Short Sleep** (19.2%), lo que sugiere el patrón de sueño más favorable entre los grupos analizados. En contraste, los participantes **Non-Hispanic Black** mostraron la mayor prevalencia de **Short Sleep** (32.8%) y la menor proporción de **Recommended Sleep** (51.6%), reflejando un patrón de sueño relativamente menos favorable. El resto de los grupos raciales y étnicos presentó distribuciones intermedias, observándose únicamente diferencias moderadas en la proporción de **Long Sleep**.

La prueba de Chi-cuadrado mostró una asociación estadísticamente significativa entre la raza/etnia y la categoría de duración del sueño (χ² = 111.06, p < 0.001). Sin embargo, el tamaño del efecto fue **débil** (V de Cramér = 0.100), lo que indica que, aunque existen diferencias entre los grupos raciales y étnicos, la raza/etnia explica únicamente una pequeña parte de la variabilidad observada en la duración del sueño dentro de la población estudiada.

### Índice de Pobreza

El Índice de Pobreza (Poverty Income Ratio, PIR) se categorizó en quintiles para facilitar el análisis de su relación con la duración del sueño. Los quintiles dividen a los participantes en cinco grupos con un tamaño de muestra aproximadamente equivalente, desde los valores más bajos hasta los más altos del índice. Un valor más alto del Poverty Index indica una mejor situación socioeconómica relativa con respecto al umbral federal de pobreza.

La distribución de las categorías de duración del sueño entre estos quintiles se analizó mediante tablas de contingencia, distribuciones porcentuales, gráficos de barras apiladas y la prueba de independencia Chi-cuadrado. La intensidad de la asociación se evaluó mediante el estadístico V de Cramér.

In [ ]:
df_analysis["Poverty_Group"] = pd.qcut(
    df_analysis["Poverty_Index"],
    q=5,
    labels=[
    "Lowest",
    "Low",
    "Middle",
    "High",
    "Highest"
    ]
)

print(df_analysis["Poverty_Group"].value_counts().sort_index())

In [ ]:
poverty_results = analyze_categorical(
    df_analysis,
    "Poverty_Group"
)

print(poverty_results["contingency"])

print(poverty_results["percentages"])

In [ ]:
poverty_results["percentages"].plot(
    kind="line",
    marker="o",
    figsize=(8,5)
)

plt.title("Sleep Duration Category by Poverty Index Quintiles")
plt.xlabel("Poverty Index Quintiles")
plt.ylabel("Percentage (%)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

colors = [
    "#ff7f0e",  # Short Sleep (orange)
    "#2ca02c",  # Recommended Sleep (green)
    "#1f77b4"   # Long Sleep (blue)
]

ax = (
    poverty_results["percentages"]
    .reindex(columns=["Short Sleep", "Recommended Sleep", "Long Sleep"])
    .plot(
        kind="bar",
        stacked=True,
        figsize=(9,6),
        width=0.8,
        color=colors
    )
)

# Add percentage labels
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        fontsize=9,
        label_type="center"
    )

plt.title("Sleep Duration Category by Poverty Index Quintiles")
plt.xlabel("Poverty Index Quintiles")
plt.ylabel("Percentage (%)")
plt.xticks(rotation=0)

plt.legend(
    title="Sleep Duration Category",
    labels=[
        "Short Sleep (<7 h)",
        "Recommended Sleep (7–9 h)",
        "Long Sleep (>9 h)"
    ],
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
print(f"Chi-square: {poverty_results['chi2']:.2f}")
print(f"Degrees of freedom: {poverty_results['dof']}")
print(f"P-value: {poverty_results['p']:.4f}")
print(f"Cramer's V: {poverty_results['cramers_v']:.3f}")

### Interpretación

El gráfico de barras apiladas muestra que la distribución de las categorías de duración del sueño varía según la situación socioeconómica de los participantes. Aquellos con un **menor Poverty Index**, es decir, con una situación económica más cercana al umbral de pobreza, presentan la menor proporción de sueño recomendado (53.7%) y la mayor proporción de sueño prolongado (20.7%).

A medida que mejora la situación socioeconómica (mayor Poverty Index), la proporción de participantes con **Recommended Sleep** tiende a aumentar, alcanzando el 68.1% en el grupo con mayor Poverty Index, aunque se observa una ligera disminución en uno de los grupos intermedios.

Por el contrario, la proporción de **Long Sleep** disminuye de forma progresiva desde el 20.7% hasta el 7.8%. En cambio, **Short Sleep** presenta pequeñas fluctuaciones entre los grupos y no muestra una tendencia claramente lineal.

En conjunto, los resultados sugieren que una mejor situación socioeconómica se asocia con una distribución más favorable de la duración del sueño, caracterizada principalmente por una mayor proporción de sueño recomendado y una menor proporción de sueño prolongado.

In [ ]:
#para presentacion
order = ["Underweight", "Normal", "Overweight", "Obese"]

plt.figure(figsize=(8,6))

sns.pointplot(
    data=df,
    x="BMI_Category",
    y="Average_Sleep_Hours",
    order=order,
    errorbar=None,
    #errorbar=("ci", 95),
    capsize=0.15,
    markers="o",
    linestyles="-"
)

plt.title("Average Sleep Duration Across BMI Categories")
plt.xlabel("")
plt.ylabel("Average Sleep Duration (hours)")

plt.grid(axis="y", alpha=0.3)

plt.show()

# Conclusiones finales

A lo largo de este proyecto se desarrolló un flujo completo de análisis de datos utilizando Python y el conjunto de datos **NHANES 2017–2018**, abarcando desde la integración y depuración de múltiples fuentes hasta el análisis exploratorio e inferencial de la información.

Los resultados obtenidos muestran que la **duración del sueño** se encuentra asociada con diferentes características demográficas, indicadores de salud y factores relacionados con el estilo de vida. Aunque la mayoría de las asociaciones detectadas presentaron un **tamaño del efecto reducido**, fueron estadísticamente significativas, lo que sugiere que el sueño forma parte de un fenómeno complejo y multifactorial en el que intervienen numerosos determinantes biológicos, sociales y conductuales.

Uno de los principales aportes del proyecto fue la creación de variables derivadas, como los **perfiles de salud** y **estilo de vida**, que permitieron sintetizar múltiples indicadores en categorías fácilmente interpretables. Esta estrategia facilitó el análisis y permitió identificar patrones que habrían sido más difíciles de observar analizando cada variable de forma aislada.

Desde un punto de vista metodológico, el proyecto puso de manifiesto la importancia de dedicar una parte significativa del trabajo a la **preparación de los datos**. La integración de distintos módulos, el tratamiento de valores perdidos, la creación de nuevas variables y la selección de criterios clínicos basados en recomendaciones internacionales fueron pasos fundamentales para garantizar la calidad y la consistencia del análisis posterior.

Es importante destacar que, debido al **diseño transversal** de NHANES, los resultados obtenidos permiten identificar **asociaciones**, pero no establecer relaciones de causalidad. Por ello, las conclusiones deben interpretarse como evidencia de patrones presentes en la población estudiada y no como demostraciones de relaciones causa-efecto.

En conjunto, este trabajo permitió aplicar de forma práctica las principales etapas de un proyecto de análisis de datos: adquisición, limpieza, transformación, exploración, visualización, análisis estadístico e interpretación de resultados. Más allá de responder a la pregunta de investigación planteada, el proyecto demuestra el valor del análisis de datos como herramienta para generar conocimiento útil a partir de grandes bases de datos poblacionales.

---

## Nota final

Con el objetivo de mantener este notebook centrado en el proceso analítico y reproducible, las interpretaciones detalladas de cada resultado, la discusión de los principales *insights*, la justificación estadística y las implicaciones prácticas de los hallazgos se desarrollan con mayor profundidad en la **presentación final del proyecto**, donde cada visualización y prueba estadística es contextualizada e interpretada dentro del marco de la investigación.